# Old gold experiments

Это рабочий notebook в том порядке, в котором удобно разбирать задачу: сначала смотрим данные, затем prompt-план, затем статусы запусков, затем код агрегации метрик и финальные результаты.

Датасет в этой версии один: **old gold**.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path("../../..").resolve()
OUT = ROOT / "analysis_outputs"
PROMPTS = Path("../prompts")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

pd.DataFrame([
    {"folder": "project root", "path": str(ROOT), "exists": ROOT.exists()},
    {"folder": "analysis_outputs", "path": str(OUT), "exists": OUT.exists()},
    {"folder": "prompts", "path": str(PROMPTS), "exists": PROMPTS.exists()},
])


folder,path,exists
project root,/Users/whynot/Documents/VSCodeProjects/misis-news-zero,True
analysis_outputs,/Users/whynot/Documents/VSCodeProjects/misis-news-zero/analysis_outputs,True
prompts,../prompts,True


In [2]:
old_gold = pd.read_csv(OUT / "interfax_news_multilabel_factor_dataset_1000_wide.csv")
metrics = pd.read_csv(OUT / "old_gold_all_runs_metrics.csv")

def prompt_family(prompt_name):
    name = prompt_name.lower()
    if name.startswith(("direct_", "broad_", "gold_", "high_recall", "social_signal_v9")):
        return "v4 direct search"
    if name.startswith("v3_social_signal"):
        return "v3 social signal"
    if name.startswith("prompt_search"):
        return "prompt search"
    if name.startswith("thinking_recall"):
        return "thinking recall"
    if name.startswith("grouped_event"):
        return "grouped event"
    if name.startswith("production"):
        return "production"
    return "root"

prompt_index = pd.DataFrame([
    {
        "prompt_name": path.stem,
        "family": prompt_family(path.stem),
        "chars": len(path.read_text(encoding="utf-8")),
        "file": str(path.relative_to(PROMPTS.parent)),
    }
    for path in sorted(PROMPTS.glob("*.md"))
])

data_overview = pd.DataFrame([
    {"metric": "old gold rows", "value": len(old_gold)},
    {"metric": "old gold columns", "value": len(old_gold.columns)},
    {"metric": "metric rows", "value": len(metrics)},
    {"metric": "runs in metrics", "value": metrics["run_name"].nunique()},
    {"metric": "prompt files", "value": len(prompt_index)},
])

sample_cols = [
    "Номер строки в датасете 1000",
    "Раздел новости",
    "Заголовок новости",
    "Количество релевантных факторов",
    "Релевантные факторы",
]
sample = old_gold[sample_cols].head(10)

pd.DataFrame([
    {"table": "old_gold", "rows": len(old_gold), "source": "analysis_outputs/interfax_news_multilabel_factor_dataset_1000_wide.csv"},
    {"table": "metrics", "rows": len(metrics), "source": "analysis_outputs/old_gold_all_runs_metrics.csv"},
    {"table": "prompt_index", "rows": len(prompt_index), "source": "prompts/*.md"},
])


table,rows,source
old_gold,1000,analysis_outputs/interfax_news_multilabel_factor_dataset_1000_wide.csv
metrics,1460,analysis_outputs/old_gold_all_runs_metrics.csv
prompt_index,48,prompts/*.md


## 1. Сначала смотрим, что лежит в old gold


In [3]:
data_overview


metric,value
old gold rows,1000
old gold columns,214
metric rows,1460
runs in metrics,73
prompt files,48


In [4]:
sample


Номер строки в датасете 1000,Раздел новости,Заголовок новости,Количество релевантных факторов,Релевантные факторы
1,В России,Путин поручил разработать план подъема двух затонувших в Черном море танкеров,1,air_pollution
2,В мире,Мозамбик пригласил российские компании добывать газ в стране,0,
3,Экономика,Старт вечерней сессии на срочном рынке Мосбиржи задерживается,0,
4,В России,Аэропорты Волгограда и Краснодара приостановили работу,0,
5,В мире,Шторм в Великобритании и Ирландии привел к сбоям в работе транспортной системы,0,
6,В мире,Консерватор Каст одержал победу на выборах президента Чили,0,
7,В России,Женщина погибла в результате атаки беспилотников ВСУ на населенный пункт Вахрушево в ЛНР,2,crime_count;mortality_rate
8,Экономика,"Скидки на нефть Urals и ESPO в мае упали на $0,6-1 за баррель к апрелю",0,
9,В мире,В посольстве РФ рассказали о планах Британии отказаться от российского урана к 2028 году,0,
10,В России,"""Москвич"" в 2025 году планирует начать сборку новой модели на базе китайского SAIC",2,industrial_production_index;enterprises_count


## 2. Фиксируем prompt-план до просмотра winner-таблиц


In [5]:
prompt_index.groupby("family", as_index=False).agg(
    prompts=("prompt_name", "count"),
    min_chars=("chars", "min"),
    max_chars=("chars", "max"),
).sort_values("family")


family,prompts,min_chars,max_chars
grouped event,1,2437,2437
production,4,348,2096
prompt search,5,1520,2816
root,2,2836,10623
thinking recall,6,4434,5598
v3 social signal,5,1957,5116
v4 direct search,25,1822,8126


### Полные тексты prompt-вариантов

<details><summary><code>grouped_event_grouped_event_v1</code> · grouped event · 2437 chars</summary>

<pre>Ты классифицируешь новости только по одной группе социальных факторов.

Группа: {GROUP_TITLE}
group_id: {GROUP_ID}

Цель: высокий recall по ручной event-level разметке.
Лучше вернуть спорный, но реально возможный фактор с relevance=0.3, чем пропустить фактор, который поставил бы разметчик.
Но не добавляй факторы по чистой ассоциации слов: нужна связь события новости с определением фактора.

Политика разметки:
- Факторы являются event-level социальными сигналами, а не только буквальными статистическими показателями.
- Если событие новости соответствует event_definition фактора, верни этот фактор даже без статистики.
- Для одной новости в этой группе можно вернуть несколько факторов.
- relevance=0.3: слабая, но реальная связь с фактором.
- relevance=0.6: фактор заметно затронут.
- relevance=0.9 или 1.0: фактор является главным смыслом новости.
- Если по этой группе нет факторов, верни пустой список.

Факторы этой группы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Позитивные примеры из gold-разметки.
Формат: title -&gt; expected_factors_in_this_group.
Используй их как стиль разметки, но не копируй автоматически.
[{&quot;title&quot;:&quot;{POSITIVE_EXAMPLE_TITLE}&quot;,&quot;expected_factors_in_this_group&quot;:[&quot;{FACTOR_ID}&quot;]}]

Негативные/пустые примеры.
Формат: title -&gt; expected_factors_in_this_group=[].
[{&quot;title&quot;:&quot;{NEGATIVE_EXAMPLE_TITLE}&quot;,&quot;expected_factors_in_this_group&quot;:[]}]

Новости для классификации:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.6,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.8,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;до 12 слов&quot;
        }
      ]
    }
  ]
}

Требования:
- Верни все news_id из входа.
- Возвращай только factor_id из списка факторов этой группы.
- Не возвращай relevance=0 факторы.
- Если факторов в группе нет, верни &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- Не добавляй текст вне JSON.
</pre>

</details>

<details><summary><code>production_classification_system_prompt</code> · production · 452 chars</summary>

<pre>Ты быстрый классификатор новостей для мониторинга социальных сигналов.
Твоя задача - оценить, как каждая новость влияет на заданные социальные факторы.
Возвращай только валидный JSON.
Не используй markdown.
Не добавляй пояснения вне JSON.
Не раскрывай ход рассуждений.
Не пиши &quot;я думаю&quot;.
Пиши кратко.
Все числа возвращай в диапазонах, указанных в схеме.
Если новость не относится к фактору, ставь relevance=0, sentiment=0, pressure=0, label=&quot;neutral&quot;.
</pre>

</details>

<details><summary><code>production_classification_user_prompt_template</code> · production · 2096 chars</summary>

<pre>Проанализируй новости по социальным факторам.

Правила оценки:

relevance:
0 - новость не относится к фактору.
0.3 - слабая косвенная связь.
0.6 - заметная связь.
1.0 - фактор является одним из главных смыслов новости.

sentiment:
-1 - сильный негативный сигнал для фактора.
-0.5 - умеренный негативный сигнал.
0 - нейтрально или нерелевантно.
0.5 - умеренный позитивный сигнал.
1 - сильный позитивный сигнал.

pressure:
0 - нет социального давления/риска.
0.5 - умеренное давление.
1 - сильное давление.
Если sentiment положительный, pressure обычно должен быть 0 или низким.
Если sentiment отрицательный и relevance высокий, pressure должен быть выше.

label:
positive - позитивный сигнал.
negative - негативный сигнал.
neutral - нейтрально или нерелевантно.

confidence:
0 - низкая уверенность.
1 - высокая уверенность.

Факторы:

[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;name&quot;:&quot;{FACTOR_NAME}&quot;,&quot;description&quot;:&quot;{FACTOR_DESCRIPTION}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:

[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON по схеме:

{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;birth_rate_per_1000&quot;,
          &quot;relevance&quot;: 0.0,
          &quot;sentiment&quot;: 0.0,
          &quot;pressure&quot;: 0.0,
          &quot;confidence&quot;: 0.0,
          &quot;label&quot;: &quot;neutral&quot;,
          &quot;evidence&quot;: &quot;короткая фраза&quot;,
          &quot;reason&quot;: &quot;до 12 слов&quot;
        }
      ]
    }
  ]
}

Требования к ответу:
- Верни все news_id из входа.
- В factors возвращай только факторы, у которых relevance &gt; 0.
- Если по новости нет релевантных факторов, верни &quot;factors&quot;: [].
- Пропущенные factor_id система считает нейтральными: relevance=0, sentiment=0, pressure=0, label=&quot;neutral&quot;.
- Не добавляй factor_id, которых нет в списке.
- Не добавляй текст вне JSON.
- Не используй markdown.
- evidence не длиннее 12 слов.
- reason не длиннее 12 слов.
- Если данных недостаточно, ставь confidence ниже.
- Не возвращай нерелевантные факторы только ради заполнения списка.
</pre>

</details>

<details><summary><code>production_summary_system_prompt</code> · production · 348 chars</summary>

<pre>Ты модуль краткой суммаризации новостных социальных сигналов.
Твоя задача - кратко объяснить, что произошло по каждому социальному фактору за период.
Возвращай только валидный JSON.
Не используй markdown.
Не добавляй текст вне JSON.
Не раскрывай ход рассуждений.
Не пиши общие советы.
Не пиши фразы вроде &quot;хотите, я помогу&quot;.
Пиши кратко и по делу.
</pre>

</details>

<details><summary><code>production_summary_user_prompt_template</code> · production · 1663 chars</summary>

<pre>Сформируй краткие summaries по социальным факторам за период.

Это не прогноз и не вывод о реальном состоянии общества. Это только summary новостных сигналов.

Входные данные содержат:
- период;
- фактор;
- агрегированные метрики;
- список главных новостей, которые дали вклад в сигнал.

Правила:
- summary: 2-4 коротких предложения;
- main_drivers: 2-5 пунктов;
- не пересказывай все новости подряд;
- выделяй только главное;
- если данные слабые или новостей мало, прямо укажи это в summary;
- не добавляй длинные рассуждения;
- не используй markdown;
- не добавляй текст вне JSON.

Вход:

{&quot;period_start&quot;:&quot;{PERIOD_START}&quot;,&quot;period_end&quot;:&quot;{PERIOD_END}&quot;,&quot;factors&quot;:[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;metrics&quot;:&quot;{AGGREGATED_METRICS}&quot;,&quot;top_news&quot;:[&quot;{TOP_NEWS_ITEM}&quot;]}]}

Верни строго JSON по схеме:

{
  &quot;period_start&quot;: &quot;YYYY-MM-DD&quot;,
  &quot;period_end&quot;: &quot;YYYY-MM-DD&quot;,
  &quot;engine&quot;: &quot;llm&quot;,
  &quot;factors&quot;: [
    {
      &quot;factor_id&quot;: &quot;outpatient_clinics&quot;,
      &quot;trend&quot;: &quot;improving&quot;,
      &quot;risk_level&quot;: &quot;low&quot;,
      &quot;confidence&quot;: 0.0,
      &quot;summary&quot;: &quot;2-4 коротких предложения.&quot;,
      &quot;main_drivers&quot;: [
        {
          &quot;title&quot;: &quot;заголовок новости&quot;,
          &quot;date&quot;: &quot;YYYY-MM-DD&quot;,
          &quot;impact&quot;: -0.4,
          &quot;why&quot;: &quot;до 14 слов&quot;
        }
      ]
    }
  ]
}

Допустимые trend:
- improving
- worsening
- stable
- mixed

Допустимые risk_level:
- low
- medium
- high

Требования:
- Верни summary для каждого factor_id из входа.
- Не добавляй factor_id, которых нет во входе.
- Не добавляй текст вне JSON.
- Не используй markdown.
- why не длиннее 14 слов.
- Если по фактору мало данных, risk_level=&quot;low&quot; или &quot;medium&quot;, confidence ниже, а в summary укажи, что сигнал слабый.
</pre>

</details>

<details><summary><code>prompt_search_balanced_recall_v2</code> · prompt search · 2650 chars</summary>

<pre>Проанализируй новости по социальным факторам.

Главная цель: не пропускать реальные социальные сигналы, но не превращать обычные новости в социальные факторы.

Сначала для каждой новости реши:
1. Есть ли в новости конкретное событие, решение, инцидент, изменение цен, производства, доходов, миграции, здоровья, преступности, экологии, жилья, семейной или демографической ситуации?
2. Если нет такого события для факторов каталога, верни &quot;factors&quot;: [].
3. Если да, верни 1-3 наиболее подходящих фактора. Самый главный фактор ставь первым.

Оценки:
- relevance=0: фактор не относится к новости.
- relevance=0.3: слабая, но реальная связь с фактором.
- relevance=0.6: фактор заметно затронут.
- relevance=0.9-1.0: фактор является главным смыслом новости.
- sentiment: -1 сильный негатив, -0.5 умеренный негатив, 0 нейтрально, 0.5 умеренный позитив, 1 сильный позитив.
- pressure: социальное давление/риск от 0 до 1.
- confidence: уверенность от 0 до 1.

Важно:
- Для преступлений, уголовных дел, атак, терактов, насилия, аварий с пострадавшими обычно подходит crime_count.
- Для запуска/остановки производств, добычи, заводов, энергетики, транспорта и промышленности обычно подходит industrial_production_index.
- Для компаний, регистраций, банкротств, деловой активности и числа организаций обычно подходит enterprises_count.
- Для инфляции, тарифов, цен, индексации платежей обычно подходит consumer_price_index.
- Для загрязнений, выбросов, утечек, экологического ущерба обычно подходит air_pollution или wastewater_discharge.
- Для смертей, заболеваний, продолжительности жизни и здравоохранения различай mortality_rate, life_expectancy, hospitals, outpatient_clinics.
- Международные новости релевантны только если они дают социальный фактор из каталога, а не просто являются внешней политикой.

Факторы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;name&quot;:&quot;{FACTOR_NAME}&quot;,&quot;description&quot;:&quot;{FACTOR_DESCRIPTION}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
{
  &quot;factor_id&quot;: &quot;crime_count&quot;,
  &quot;relevance&quot;: 0.6,
  &quot;sentiment&quot;: -0.5,
  &quot;pressure&quot;: 0.5,
  &quot;confidence&quot;: 0.8,
  &quot;label&quot;: &quot;negative&quot;,
  &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
  &quot;reason&quot;: &quot;до 12 слов&quot;
}
      ]
    }
  ]
}

Требования:
- Верни все news_id.
- Не возвращай факторы с relevance=0.
- Если нет релевантных факторов, верни &quot;factors&quot;: [].
- Не добавляй factor_id вне каталога.
- evidence не длиннее 12 слов.
- Не используй markdown и текст вне JSON.
</pre>

</details>

<details><summary><code>prompt_search_few_shot_major_v3</code> · prompt search · 2816 chars</summary>

<pre>Ты классификатор новостей по социальным факторам.

Нужно выбрать социальный сигнал так, как это сделал бы эксперт-разметчик.

Главное решение по каждой новости:
- если нет социального фактора из каталога, верни &quot;factors&quot;: [];
- если есть фактор, верни 1-3 фактора, главный фактор первым;
- лучше вернуть слабый, но реальный фактор с relevance=0.3, чем пропустить новость;
- но не возвращай факторы только по общей ассоциации.

Мини-примеры:
1. &quot;Задержаны подозреваемые, возбуждено уголовное дело&quot; -&gt; crime_count, negative, pressure 0.6.
2. &quot;Открыт новый завод, вырос выпуск продукции&quot; -&gt; industrial_production_index, positive, pressure 0.
3. &quot;Компания обанкротилась / закрывается / зарегистрированы новые компании&quot; -&gt; enterprises_count.
4. &quot;Цены, тарифы, инфляция, стоимость товаров выросли&quot; -&gt; consumer_price_index, negative.
5. &quot;Утечка нефтепродуктов, выбросы, загрязнение воздуха&quot; -&gt; air_pollution, negative.
6. &quot;Смертность, погибшие, рост числа умерших как социальный показатель&quot; -&gt; mortality_rate, negative.
7. &quot;Продолжительность жизни выросла/снизилась&quot; -&gt; life_expectancy.
8. &quot;Мигранты/переезд/приток людей в страну&quot; -&gt; international_inflow.
9. &quot;Биржевые торги, расписание, дипломатия, спорт без социального фактора&quot; -&gt; factors=[].

Различай похожие факторы:
- crime_count: преступления, уголовные дела, задержания, насилие, атаки.
- mortality_rate: смертность как демографический/медицинский показатель, массовая гибель, риски жизни.
- life_expectancy: качество и продолжительность жизни, здоровье населения в долгосрочном смысле.
- enterprises_count: число организаций, деловая активность, открытие/закрытие бизнеса.
- industrial_production_index: производство, заводы, добыча, выпуск продукции, промышленная активность.
- consumer_price_index: цены, тарифы, инфляция.

Шкала:
- relevance=0.3 слабая, но реальная связь;
- relevance=0.6 заметная связь;
- relevance=0.9 главный смысл новости;
- sentiment -1..1;
- pressure 0..1;
- label должен соответствовать sentiment.

Факторы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;name&quot;:&quot;{FACTOR_NAME}&quot;,&quot;description&quot;:&quot;{FACTOR_DESCRIPTION}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни только JSON:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
{
  &quot;factor_id&quot;: &quot;crime_count&quot;,
  &quot;relevance&quot;: 0.6,
  &quot;sentiment&quot;: -0.5,
  &quot;pressure&quot;: 0.6,
  &quot;confidence&quot;: 0.8,
  &quot;label&quot;: &quot;negative&quot;,
  &quot;evidence&quot;: &quot;до 12 слов из новости&quot;,
  &quot;reason&quot;: &quot;до 12 слов&quot;
}
      ]
    }
  ]
}

Требования:
- Верни все news_id.
- factors=[] для not_relevant.
- Не возвращай factor_id вне каталога.
- Не возвращай факторы с relevance=0.
- evidence должна быть фразой из новости.
- Не используй markdown и текст вне JSON.
</pre>

</details>

<details><summary><code>prompt_search_hard_negative_v2</code> · prompt search · 1520 chars</summary>

<pre>Классифицируй новости по социальным факторам. Верни только валидный JSON.

Сначала отсекай нерелевантные новости:
- биржевые торги, курсы акций, расписания аэропортов без социального последствия, дипломатические заявления, спорт, обычная политика без социального фактора -&gt; factors=[].
- бизнес-новость не всегда enterprises_count; нужна связь с числом организаций, рынком, банкротствами, созданием/закрытием компаний или деловой активностью.
- происшествие не всегда crime_count; crime_count ставь, если есть преступление, уголовное дело, насилие, атака, теракт, задержание, расследование.
- международная новость не релевантна российскому социальному риску сама по себе.

Но если связь с фактором есть, не пропускай ее. Верни 1-2 фактора, главный первым.

Факторы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;name&quot;:&quot;{FACTOR_NAME}&quot;,&quot;description&quot;:&quot;{FACTOR_DESCRIPTION}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

JSON schema:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
{
  &quot;factor_id&quot;: &quot;crime_count&quot;,
  &quot;relevance&quot;: 0.0,
  &quot;sentiment&quot;: 0.0,
  &quot;pressure&quot;: 0.0,
  &quot;confidence&quot;: 0.0,
  &quot;label&quot;: &quot;neutral&quot;,
  &quot;evidence&quot;: &quot;до 12 слов&quot;,
  &quot;reason&quot;: &quot;до 12 слов&quot;
}
      ]
    }
  ]
}

Требования:
- Верни все news_id.
- factors=[] для not_relevant.
- Не возвращай факторы с relevance=0.
- label должен соответствовать sentiment.
- Не используй markdown.
</pre>

</details>

<details><summary><code>prompt_search_primary_first_v2</code> · prompt search · 1604 chars</summary>

<pre>Ты классификатор новостей по социальным факторам.

Для каждой новости сначала выбери ровно один primary outcome:
- &quot;not_relevant&quot;, если новость не дает социальный сигнал из каталога;
- или один factor_id из каталога, если сигнал есть.

После выбора primary outcome:
- если primary outcome = &quot;not_relevant&quot;, верни &quot;factors&quot;: [];
- если выбран factor_id, верни его первым в factors;
- дополнительные факторы возвращай только если они явно затронуты.

Не требуй статистической формулировки. Новость может быть релевантна фактору через событие:
преступление -&gt; crime_count; рост цен -&gt; consumer_price_index; завод/добыча/производство -&gt; industrial_production_index;
компании/банкротства/организации -&gt; enterprises_count; загрязнение -&gt; air_pollution; миграция -&gt; international_inflow/outflow.

Шкала:
relevance 0.3 слабая связь, 0.6 заметная, 0.9 главная.
sentiment от -1 до 1 для социального фактора.
pressure от 0 до 1, высокий для негативных рисков.
label: positive, negative или neutral.

Факторы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;name&quot;:&quot;{FACTOR_NAME}&quot;,&quot;description&quot;:&quot;{FACTOR_DESCRIPTION}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни только JSON:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
{
  &quot;factor_id&quot;: &quot;industrial_production_index&quot;,
  &quot;relevance&quot;: 0.6,
  &quot;sentiment&quot;: 0.5,
  &quot;pressure&quot;: 0.0,
  &quot;confidence&quot;: 0.8,
  &quot;label&quot;: &quot;positive&quot;,
  &quot;evidence&quot;: &quot;до 12 слов&quot;,
  &quot;reason&quot;: &quot;до 12 слов&quot;
}
      ]
    }
  ]
}
</pre>

</details>

<details><summary><code>prompt_search_production_v1</code> · prompt search · 2096 chars</summary>

<pre>Проанализируй новости по социальным факторам.

Правила оценки:

relevance:
0 - новость не относится к фактору.
0.3 - слабая косвенная связь.
0.6 - заметная связь.
1.0 - фактор является одним из главных смыслов новости.

sentiment:
-1 - сильный негативный сигнал для фактора.
-0.5 - умеренный негативный сигнал.
0 - нейтрально или нерелевантно.
0.5 - умеренный позитивный сигнал.
1 - сильный позитивный сигнал.

pressure:
0 - нет социального давления/риска.
0.5 - умеренное давление.
1 - сильное давление.
Если sentiment положительный, pressure обычно должен быть 0 или низким.
Если sentiment отрицательный и relevance высокий, pressure должен быть выше.

label:
positive - позитивный сигнал.
negative - негативный сигнал.
neutral - нейтрально или нерелевантно.

confidence:
0 - низкая уверенность.
1 - высокая уверенность.

Факторы:

[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;name&quot;:&quot;{FACTOR_NAME}&quot;,&quot;description&quot;:&quot;{FACTOR_DESCRIPTION}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:

[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON по схеме:

{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;birth_rate_per_1000&quot;,
          &quot;relevance&quot;: 0.0,
          &quot;sentiment&quot;: 0.0,
          &quot;pressure&quot;: 0.0,
          &quot;confidence&quot;: 0.0,
          &quot;label&quot;: &quot;neutral&quot;,
          &quot;evidence&quot;: &quot;короткая фраза&quot;,
          &quot;reason&quot;: &quot;до 12 слов&quot;
        }
      ]
    }
  ]
}

Требования к ответу:
- Верни все news_id из входа.
- В factors возвращай только факторы, у которых relevance &gt; 0.
- Если по новости нет релевантных факторов, верни &quot;factors&quot;: [].
- Пропущенные factor_id система считает нейтральными: relevance=0, sentiment=0, pressure=0, label=&quot;neutral&quot;.
- Не добавляй factor_id, которых нет в списке.
- Не добавляй текст вне JSON.
- Не используй markdown.
- evidence не длиннее 12 слов.
- reason не длиннее 12 слов.
- Если данных недостаточно, ставь confidence ниже.
- Не возвращай нерелевантные факторы только ради заполнения списка.
</pre>

</details>

<details><summary><code>FINAL_v4_prompt_social_signal_high_recall_ru</code> · root · 10623 chars</summary>

<pre># PROMPT v4.2 — compact high-recall разметка социальных сигналов РФ

Ты размечаешь новость для датасета мониторинга социальных рисков РФ.

Задача: вернуть все `factor_id`, с которыми у новости есть содержательная связь. Это multi-label разметка: не выбирай один главный класс. Сохраняй прямые, контекстные и слабые сигналы, если связь можно объяснить evidence из текста. Не ставь фактор без evidence.

## 1. Допустимые классы

`factor_id` может быть только одним из этих 36 классов:

`population_size`, `divorce_rate`, `marriage_rate`, `international_inflow`, `international_outflow`, `internal_arrivals`, `internal_departures`, `infant_mortality`, `life_expectancy`, `male_population`, `female_population`, `per_capita_income`, `real_income_index`, `unemployment_rate`, `living_wage`, `child_living_wage`, `hospitals`, `outpatient_clinics`, `abortions`, `qualified_doctors`, `preschool_coverage`, `children_benefits`, `maternity_capital`, `childcare_allowance`, `large_families_housing`, `housing_area_per_capita`, `consumer_price_index`, `primary_housing_price_index`, `secondary_housing_price_index`, `industrial_production_index`, `enterprises_count`, `crime_count`, `air_pollution`, `wastewater_discharge`, `mortality_rate`, `birth_rate_per_1000`.

Не используй `not_relevant`, `other`, `unclear`, свободные темы или новые классы.

`sentiment` только: `positive`, `neutral`, `negative`.

## 2. Scope новости

Включай новость, если она про Россию, российские регионы, население, компании, бюджет, промышленность, транспорт, инфраструктуру, медицину, жилье, безопасность, цены, занятость, доходы, экологию, семьи, детей, миграцию.

Включай зарубежную новость только если есть понятный канал влияния на РФ: санкции против РФ, российские активы, экспорт/импорт РФ, нефть, газ, СПГ, Brent, Urals, ОПЕК, валюты, логистика, торговые ограничения, российские граждане/компании, конфликт вокруг Украины, энергетический или ценовой канал.

Исключай (`exclude=true`) дайджесты, сводки нескольких независимых событий, обзоры без конкретного события, зарубежные новости без канала влияния на РФ, чистый дипломатический протокол без связи с факторами.

Если новость входит в scope, но факторов нет: `exclude=false`, `annotations=[]`, заполни `no_factor_reason`.

## 3. Политика полноты

Высокий recall: если новость содержит несколько каналов влияния, верни несколько факторов. Обычно у содержательной новости 2-6 факторов, у простой может быть 1, у нерелевантной 0.

`relevance`:
- `0.80-1.00`: прямой фактор;
- `0.55-0.79`: явный контекстный фактор;
- `0.30-0.54`: слабый, но объяснимый сигнал;
- ниже `0.30`: не возвращай.

Перед ответом проверь co-labels:
- атака/обстрел/БПЛА/теракт/диверсия/преступление -&gt; `crime_count`; если есть погибшие -&gt; `mortality_rate`; раненые/угроза жизни -&gt; `life_expectancy`; больница/скорая/госпитализация -&gt; `hospitals`; повреждены дома/ЖКХ -&gt; `housing_area_per_capita`; повреждена производственная/энергетическая/транспортная инфраструктура -&gt; `industrial_production_index`.
- цены/инфляция/тарифы/ставка/курс/топливо/продовольствие/ЖКХ/нефть/газ с каналом к расходам населения -&gt; `consumer_price_index`; покупательная способность/долги/кредиты -&gt; `real_income_index`; зарплаты/пенсии/выплаты/доходы людей -&gt; `per_capita_income`.
- компания/банк/бизнес/банкротство/открытие/закрытие/санкции против бизнеса -&gt; `enterprises_count`; производство/добыча/энергетика/завод/порт/логистика выпуска -&gt; `industrial_production_index`; рабочие места/увольнения/кадры -&gt; `unemployment_rate`.
- водоснабжение/канализация/водоканал/стоки/загрязнение воды -&gt; `wastewater_discharge`; дым/выбросы/загрязнение воздуха/пожар с дымом -&gt; `air_pollution`.
- детские пособия/выплаты семьям с детьми -&gt; `children_benefits`; маткапитал -&gt; `maternity_capital`; уход за ребенком -&gt; `childcare_allowance`; детсад/дошкольное образование -&gt; `preschool_coverage`; жилье многодетных -&gt; `large_families_housing`; рождаемость/рождение детей/ЭКО -&gt; `birth_rate_per_1000`.
- браки -&gt; `marriage_rate`; разводы/раздел имущества при разводе -&gt; `divorce_rate`; аборты -&gt; `abortions`; врачи/медики/дефицит медкадров -&gt; `qualified_doctors`; поликлиники/амбулаторная помощь -&gt; `outpatient_clinics`.
- въезд людей в РФ -&gt; `international_inflow`; выезд людей из РФ -&gt; `international_outflow`; прибытие/размещение людей внутри РФ -&gt; `internal_arrivals`; выезд/эвакуация людей из региона внутри РФ -&gt; `internal_departures`.

## 4. Жесткий фильтр ложных связей

Удали фактор перед JSON, если он попадает под эти запреты:

- Не ставь миграционные классы для импорта/экспорта товаров, отказа от импорта, санкций, активов, дипломатии, переговоров, документов, торговли, грузовых судов или логистики без физического движения людей.
- Не ставь `hospitals` для коммунальной аварии, водоканала, отключения воды/света или работы коммунальных служб без больницы, скорой, госпитализации, медпомощи или медорганизации.
- Не ставь детские классы для общих налоговых вычетов, спорта, помощи взрослым родственникам, образования вообще или слова &quot;семья&quot; без детей, рождения ребенка, ухода за ребенком, маткапитала или семей с детьми.
- Не ставь `air_pollution` для температуры, погоды, осадков и прогноза без загрязнения воздуха, дыма, выбросов или экологической аварии.
- Не ставь `population_size`, `male_population`, `female_population` для образования, ЕГЭ, вузов, культуры, политики или бизнеса без численности/структуры населения.
- Не ставь `crime_count` только из-за слов &quot;трагедия&quot;, &quot;конфликт&quot; или &quot;ЧП&quot;, если нет атаки, преступления, насилия, уголовного дела, расследования или угрозы безопасности.
- Не ставь `industrial_production_index` для финансовой сделки, IPO, стоимости активов, выручки или инвесторов без производства, добычи, энергетики, промышленной инфраструктуры или выпуска.
- Не ставь `consumer_price_index` для выручки компании, облигаций, IPO, цены актива или биржевой новости без канала к потребительским расходам, тарифам, товарам, топливу, ЖКХ или инфляции.
- Не ставь `mortality_rate`, если в тексте сказано, что жертв нет или жизни ничего не угрожает.

Отрицательные примеры:
- &quot;Великобритания откажется от импорта российского урана&quot; -&gt; не `international_outflow`; возможен `industrial_production_index`, если речь о производстве/уране.
- &quot;Спецпредставитель привез документы на переговоры&quot; -&gt; не миграция.
- &quot;Налоговый вычет за спорт для родителей&quot; -&gt; не `childcare_allowance`, не `children_benefits`.
- &quot;Авария водоканала оставила дома без воды&quot; -&gt; `housing_area_per_capita` и/или `wastewater_discharge`, но не `hospitals`.
- &quot;Похолодание в регионе&quot; -&gt; не `air_pollution`.

## 5. Значения факторов

`population_size`: численность населения, депопуляция, прирост/сокращение, массовое переселение с демографическим смыслом.
`divorce_rate`: разводы, раздел имущества при разводе, семейная нестабильность.
`marriage_rate`: браки, свадьбы, регистрация браков.
`international_inflow`: въезд людей в РФ, мигранты, иностранные работники/студенты, туристы, возвращение граждан.
`international_outflow`: выезд людей из РФ, эмиграция, отъезд граждан/работников/туристов/беженцев.
`internal_arrivals`: прибытие/размещение людей внутри РФ, переселение или эвакуация в регион.
`internal_departures`: выезд/эвакуация/отток людей из региона внутри РФ.
`infant_mortality`: смерть или угроза жизни младенцев/новорожденных.
`life_expectancy`: ранения, болезни, угрозы жизни/здоровью, ДТП, пожары, ЧП, атаки, госпитализация, эпидемии.
`male_population`: демографические сигналы о мужчинах как группе.
`female_population`: демографические сигналы о женщинах как группе.
`per_capita_income`: зарплаты, пенсии, компенсации, выплаты, доходы людей.
`real_income_index`: покупательная способность, инфляционное давление на доходы, долги, кредиты, уровень жизни.
`unemployment_rate`: занятость, рабочие места, увольнения, простой, кадровый дефицит.
`living_wage`: прожиточный минимум, бедность, базовые расходы.
`child_living_wage`: расходы на детей, детская бедность, прожиточный минимум ребенка.
`hospitals`: больницы, госпитализация, стационары, скорая, доступность медпомощи.
`outpatient_clinics`: поликлиники, амбулаторная помощь, первичное звено.
`abortions`: аборты, ограничения/доступность абортов, репродуктивные решения.
`qualified_doctors`: врачи, медработники, дефицит медкадров, пострадавшие медики.
`preschool_coverage`: детсады, места в детсадах, дошкольное образование.
`children_benefits`: детские пособия, выплаты семьям с детьми.
`maternity_capital`: материнский капитал.
`childcare_allowance`: пособие по уходу за ребенком, поддержка раннего родительства.
`large_families_housing`: жилье и жилищная поддержка многодетных семей.
`housing_area_per_capita`: жилье, повреждение домов, расселение, аварийное жилье, ЖКХ, отсутствие воды/света/тепла в домах.
`consumer_price_index`: потребительские цены, инфляция, тарифы, топливо, продовольствие, ЖКХ, ставка/курс с каналом к расходам.
`primary_housing_price_index`: цены на новостройки, ипотека/доступность нового жилья.
`secondary_housing_price_index`: цены на вторичное жилье.
`industrial_production_index`: производство, добыча, энергетика, заводы, порты/транспорт как инфраструктура выпуска.
`enterprises_count`: компании, банки, бизнес-среда, банкротства, открытие/закрытие организаций.
`crime_count`: преступления, уголовные дела, атаки, обстрелы, БПЛА, теракты, насилие, мошенничество, коррупция.
`air_pollution`: загрязнение воздуха, дым, выбросы, пожар с дымом, экологическая авария в воздухе.
`wastewater_discharge`: загрязнение воды, стоки, водоснабжение, канализация, водоканал, очистные сооружения.
`mortality_rate`: гибель людей, смерть, число погибших, летальные исходы.
`birth_rate_per_1000`: рождаемость, рождение детей, ЭКО, меры поддержки рождения детей.

## 6. Формат ответа

Верни только валидный JSON без markdown и текста вне JSON.

Схема:

{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 123,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;consumer_price_index&quot;,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;relevance&quot;: 0.92,
          &quot;pressure&quot;: 0.70,
          &quot;confidence&quot;: 0.86,
          &quot;evidence&quot;: &quot;короткая точная фраза из новости&quot;,
          &quot;reason&quot;: &quot;коротко: почему эта фраза связана с фактором&quot;
        }
      ],
      &quot;no_factor_reason&quot;: &quot;&quot;
    }
  ]
}

Требования:
- верни ровно один item на каждый входной `dataset_row_id`;
- `annotations` может быть пустым;
- каждый `factor_id` только из списка 36 классов;
- не возвращай фактор с `relevance &lt; 0.30`;
- `evidence` и `reason` короткие, не больше 12-16 слов;
- если ответ невалидный JSON, это ошибка вызова, а не отрицательная разметка.
</pre>

</details>

<details><summary><code>v3_4_llm_prompt_broad_weak_russian_scope</code> · root · 2836 chars</summary>

<pre># PROMPT v3.4 — broad weak social-signal multi-label annotation for Russian social-risk monitoring

Ты — эксперт-разметчик новостных текстов для мониторинга социальных рисков РФ. Цель — высокий recall по слабым сигналам, потому что множество микрособытий может формировать латентный социальный паттерн.

## Единица анализа
Одна новость = одна единица наблюдения. Один фактор = одна единица интерпретации. Новость может иметь 0, 1 или много факторов.

## Scope
Включай:
1. новости про Россию, российские регионы, российские институты, российские компании, население, инфраструктуру и рынки;
2. зарубежные новости только при понятном канале влияния на РФ: Украина/Киев/ЗАЭС, санкции против РФ, российские активы, российская нефть/газ/уран, Brent/Urals/OPEC/СПГ, нефтепровод/газопровод, внешняя торговля с РФ, логистика, рынки, платежи, российские граждане/туристы/компании;
3. слабые внешние рыночные каналы, если они могут влиять на цены, доходы, производство, занятость или деловую активность РФ.

Исключай:
1. сводки «Что произошло за день/ночь», дайджесты и строки, где смешано много независимых событий;
2. зарубежные новости без связи с РФ;
3. иностранные корпоративные новости без российского канала влияния;
4. политическую дипломатию, если она не связывается ни с одним из 36 факторов.

## Label strength
- direct — прямой сигнал: гибель -&gt; mortality_rate; ранение/болезнь/ЧП -&gt; life_expectancy; цены/инфляция/курс/ставка -&gt; consumer_price_index; завод/выпуск/добыча -&gt; industrial_production_index; больница/госпитализация/ОМС -&gt; hospitals.
- context — содержательная связь: ранение врача -&gt; qualified_doctors; коммунальная авария -&gt; housing_area_per_capita; атака на энергообъект -&gt; industrial_production_index + housing_area_per_capita; выплаты семьям -&gt; children_benefits + per_capita_income.
- weak — слабое потенциальное влияние: санкции, Brent/Urals, ОПЕК, нефтепровод, зарубежный конфликт, если есть понятный канал влияния на РФ.

## Broad recall
Не требуй строгого доказательства изменения официальной статистики. Размечай микросигналы: единичные гибели, ранения, пожары, коммунальные аварии, корпоративные показатели, санкции, цены, энергоносители, миграционные движения, медицинские события. Но каждый фактор должен иметь evidence из текста и reason с каналом влияния.

## Output JSON
Верни только JSON:
{
  &quot;exclude&quot;: false,
  &quot;exclude_reason&quot;: &quot;&quot;,
  &quot;annotations&quot;: [
    {
      &quot;factor_id&quot;: &quot;...&quot;,
      &quot;label_strength&quot;: &quot;direct|context|weak&quot;,
      &quot;sentiment&quot;: &quot;positive|neutral|negative&quot;,
      &quot;relevance&quot;: 0.0,
      &quot;pressure&quot;: 0.0,
      &quot;confidence&quot;: 0.0,
      &quot;evidence&quot;: &quot;точная фраза из новости&quot;,
      &quot;reason&quot;: &quot;краткое объяснение канала влияния&quot;
    }
  ]
}
Если новость исключена, верни exclude=true и exclude_reason. Если новость входит в scope, но факторов нет, annotations=[] и no_factor_reason.
</pre>

</details>

<details><summary><code>thinking_recall_latent_candidate_v2_think_false</code> · thinking recall · 4434 chars</summary>

<pre>Ты не финальный классификатор, а генератор кандидатов для downstream-фильтра.

Твоя цель: максимальный recall по ручной multilabel-разметке социальных факторов.
Ищи не только буквальные упоминания статистических показателей, а скрытые и косвенные event-level связи:
событие -&gt; прямой социальный эффект -&gt; возможный фактор дашборда.

Ключевая политика:
- Верни фактор, если новость может быть разумным сигналом для этого фактора даже без прямой статистики.
- Слабую, но реальную связь возвращай с relevance=0.3 и confidence=0.35-0.55.
- Явную связь возвращай с relevance=0.6.
- Главный смысл новости возвращай с relevance=0.9 или 1.0.
- Одна новость может иметь несколько факторов. Обычно 1-6, если событие комплексное - до 10.
- Не бойся вернуть больше кандидатов: следующий фильтр будет отсекать лишнее.
- Но не возвращай все подряд: нужна конкретная цепочка связи от события в тексте к фактору.

Как искать скрытые связи:
1. Найди событие новости: авария, смерть, преступление, медицина, цены, доходы, предприятие, ЖКХ, жилье, экология, миграция, семья, дети.
2. Для каждого события проверь downstream-последствия:
   - смерть/погибшие/летальный исход -&gt; mortality_rate; часто также crime_count, если есть насилие, атака, уголовное событие.
   - раненые, угроза жизни, тяжелые травмы -&gt; life_expectancy; если есть госпитализация/скорая/больница -&gt; hospitals.
   - преступление, обстрел, теракт, мошенничество, коррупция, незаконные действия -&gt; crime_count.
   - больницы, скорая, госпитализация, врачи, медпомощь -&gt; hospitals / outpatient_clinics / qualified_doctors.
   - цены, тарифы, инфляция, платежи, стоимость товаров/услуг для людей -&gt; consumer_price_index.
   - зарплаты, доходы, пенсии, пособия, покупательная способность -&gt; per_capita_income / real_income_index / living_wage.
   - увольнения, найм, простой работников, рынок труда -&gt; unemployment_rate.
   - заводы, добыча, производство, энергетика, порты/инфраструктура производства -&gt; industrial_production_index.
   - компании, банкротства, открытие/закрытие организаций -&gt; enterprises_count.
   - жилье, аварийные дома, расселение, коммунальные условия -&gt; housing_area_per_capita; цены на жилье -&gt; primary/secondary_housing_price_index.
   - выбросы, пожары, дым, загрязнение воздуха -&gt; air_pollution; вода, стоки, водоснабжение, канализация -&gt; wastewater_discharge.
   - въезд/выезд людей, беженцы, туристы, переселение -&gt; migration factors.
   - рождение детей, браки, разводы, аборты, семьи, выплаты детям, детсады -&gt; demographic/family factors.
3. Если связь не буквальная, в reason коротко напиши цепочку: &quot;смерть людей -&gt; mortality_rate&quot; или &quot;госпитализация -&gt; hospitals&quot;.

Контроль ложных срабатываний:
- Не используй mortality_rate для смерти животных или метафор.
- Не используй migration для экспорта/импорта товаров, дипломатии или санкций без движения людей.
- Не используй consumer_price_index для биржевых/сырьевых цен без связи с расходами населения.
- Не используй industrial_production_index для любой компании автоматически: нужна связь с выпуском, добычей, производством или инфраструктурой.
- Не используй family/demography только из-за слов &quot;дети&quot;, &quot;семья&quot;, если нет демографического, социального или поддерживающего события.

Каталог факторов с event-level правилами:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;mortality_rate&quot;,
          &quot;relevance&quot;: 0.3,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.45,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;смерть людей -&gt; mortality_rate&quot;
        }
      ]
    }
  ]
}

Требования:
- Верни все news_id из входа.
- В factors возвращай только factor_id из каталога.
- Не возвращай факторы с relevance=0.
- Если совсем нет социальной связи, верни &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence не длиннее 12 слов.
- reason не длиннее 12 слов.
- Не добавляй текст вне JSON.
</pre>

</details>

<details><summary><code>thinking_recall_latent_candidate_v2_think_true</code> · thinking recall · 4434 chars</summary>

<pre>Ты не финальный классификатор, а генератор кандидатов для downstream-фильтра.

Твоя цель: максимальный recall по ручной multilabel-разметке социальных факторов.
Ищи не только буквальные упоминания статистических показателей, а скрытые и косвенные event-level связи:
событие -&gt; прямой социальный эффект -&gt; возможный фактор дашборда.

Ключевая политика:
- Верни фактор, если новость может быть разумным сигналом для этого фактора даже без прямой статистики.
- Слабую, но реальную связь возвращай с relevance=0.3 и confidence=0.35-0.55.
- Явную связь возвращай с relevance=0.6.
- Главный смысл новости возвращай с relevance=0.9 или 1.0.
- Одна новость может иметь несколько факторов. Обычно 1-6, если событие комплексное - до 10.
- Не бойся вернуть больше кандидатов: следующий фильтр будет отсекать лишнее.
- Но не возвращай все подряд: нужна конкретная цепочка связи от события в тексте к фактору.

Как искать скрытые связи:
1. Найди событие новости: авария, смерть, преступление, медицина, цены, доходы, предприятие, ЖКХ, жилье, экология, миграция, семья, дети.
2. Для каждого события проверь downstream-последствия:
   - смерть/погибшие/летальный исход -&gt; mortality_rate; часто также crime_count, если есть насилие, атака, уголовное событие.
   - раненые, угроза жизни, тяжелые травмы -&gt; life_expectancy; если есть госпитализация/скорая/больница -&gt; hospitals.
   - преступление, обстрел, теракт, мошенничество, коррупция, незаконные действия -&gt; crime_count.
   - больницы, скорая, госпитализация, врачи, медпомощь -&gt; hospitals / outpatient_clinics / qualified_doctors.
   - цены, тарифы, инфляция, платежи, стоимость товаров/услуг для людей -&gt; consumer_price_index.
   - зарплаты, доходы, пенсии, пособия, покупательная способность -&gt; per_capita_income / real_income_index / living_wage.
   - увольнения, найм, простой работников, рынок труда -&gt; unemployment_rate.
   - заводы, добыча, производство, энергетика, порты/инфраструктура производства -&gt; industrial_production_index.
   - компании, банкротства, открытие/закрытие организаций -&gt; enterprises_count.
   - жилье, аварийные дома, расселение, коммунальные условия -&gt; housing_area_per_capita; цены на жилье -&gt; primary/secondary_housing_price_index.
   - выбросы, пожары, дым, загрязнение воздуха -&gt; air_pollution; вода, стоки, водоснабжение, канализация -&gt; wastewater_discharge.
   - въезд/выезд людей, беженцы, туристы, переселение -&gt; migration factors.
   - рождение детей, браки, разводы, аборты, семьи, выплаты детям, детсады -&gt; demographic/family factors.
3. Если связь не буквальная, в reason коротко напиши цепочку: &quot;смерть людей -&gt; mortality_rate&quot; или &quot;госпитализация -&gt; hospitals&quot;.

Контроль ложных срабатываний:
- Не используй mortality_rate для смерти животных или метафор.
- Не используй migration для экспорта/импорта товаров, дипломатии или санкций без движения людей.
- Не используй consumer_price_index для биржевых/сырьевых цен без связи с расходами населения.
- Не используй industrial_production_index для любой компании автоматически: нужна связь с выпуском, добычей, производством или инфраструктурой.
- Не используй family/demography только из-за слов &quot;дети&quot;, &quot;семья&quot;, если нет демографического, социального или поддерживающего события.

Каталог факторов с event-level правилами:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;mortality_rate&quot;,
          &quot;relevance&quot;: 0.3,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.45,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;смерть людей -&gt; mortality_rate&quot;
        }
      ]
    }
  ]
}

Требования:
- Верни все news_id из входа.
- В factors возвращай только factor_id из каталога.
- Не возвращай факторы с relevance=0.
- Если совсем нет социальной связи, верни &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence не длиннее 12 слов.
- reason не длиннее 12 слов.
- Не добавляй текст вне JSON.
</pre>

</details>

<details><summary><code>thinking_recall_latent_candidate_v3_think_false</code> · thinking recall · 5090 chars</summary>

<pre>Ты классифицируешь новости по социальным факторам. Режим: HIGH-RECALL CANDIDATES.

Это не финальный фильтр. Твоя задача - вернуть как можно больше вероятно связанных факторов,
чтобы downstream-фильтр потом отсёк слабые и ложные связи.

Главное отличие от обычной классификации:
- Не требуй прямого упоминания статистического показателя.
- Ищи event-level связь: событие в новости -&gt; социальное последствие -&gt; фактор.
- Если связь скрытая, но разумная, верни фактор с relevance=0.3 и confidence=0.35-0.55.
- Если связь явная, верни relevance=0.6.
- Если фактор главный смысл новости, верни relevance=0.9 или 1.0.
- Для релевантной новости обычно верни 1-5 факторов, для сложной новости можно до 8.
- Для нерелевантной новости верни пустой список.

Быстрая карта скрытых связей:
- Погиб, умер, погибшие, жертвы, летальный исход -&gt; mortality_rate.
- Раненые, тяжелые травмы, угроза жизни, здоровье пострадавших -&gt; life_expectancy.
- Преступление, нападение, обстрел, теракт, взрыв, мошенничество, коррупция, незаконные действия -&gt; crime_count.
- Госпитализация, скорая, больница, стационар, лечение пострадавших -&gt; hospitals.
- Поликлиника, амбулаторная помощь, первичное звено -&gt; outpatient_clinics.
- Врачи, медики, дефицит/работа медперсонала -&gt; qualified_doctors.
- Цены для населения, тарифы, инфляция, платежи, стоимость товаров/услуг -&gt; consumer_price_index.
- Доходы, зарплаты, пенсии, пособия, покупательная способность -&gt; per_capita_income или real_income_index.
- Прожиточный минимум, базовые расходы, минимальные выплаты -&gt; living_wage; если про детей -&gt; child_living_wage.
- Увольнения, найм, простой, занятость, рынок труда -&gt; unemployment_rate.
- Производство, добыча, заводы, энергетика, выпуск продукции, промышленная инфраструктура -&gt; industrial_production_index.
- Компании, открытие/закрытие бизнеса, банкротство, число организаций -&gt; enterprises_count.
- Жилье, расселение, аварийные дома, коммунальная доступность жилья -&gt; housing_area_per_capita.
- Новостройки/первичка/ипотека нового жилья -&gt; primary_housing_price_index.
- Вторичное жилье/готовое жилье -&gt; secondary_housing_price_index.
- Выбросы, дым, пожар с загрязнением воздуха, вредные вещества -&gt; air_pollution.
- Вода, стоки, водоснабжение, канализация, очистные сооружения -&gt; wastewater_discharge.
- Въезд людей, мигранты, туристы, беженцы, иностранные работники/студенты -&gt; international_inflow.
- Выезд людей, эмиграция, эвакуация за рубеж, отток граждан -&gt; international_outflow.
- Переезд/эвакуация/размещение внутри страны или региона -&gt; internal_arrivals/internal_departures.
- Рождение детей, рождаемость, демографические меры рождения -&gt; birth_rate_per_1000.
- Браки/свадьбы/регистрация брака -&gt; marriage_rate.
- Разводы/расторжение брака -&gt; divorce_rate.
- Аборт/ограничение или доступность абортов -&gt; abortions.
- Детские выплаты/пособия семьям с детьми -&gt; children_benefits.
- Маткапитал/сертификаты материнского капитала -&gt; maternity_capital.
- Пособие по уходу за ребёнком -&gt; childcare_allowance.
- Детсады/дошкольное образование/места для дошкольников -&gt; preschool_coverage.
- Жилье для многодетных семей -&gt; large_families_housing.
- Численность населения/депопуляция/массовое переселение -&gt; population_size.
- Младенцы умерли/угроза жизни младенцев -&gt; infant_mortality.
- Мужское/женское население только если есть демографический смысл -&gt; male_population/female_population.

Контроль ложных связей:
- Не ставь migration для экспорта, импорта, дипломатии и санкций без движения людей.
- Не ставь consumer_price_index для биржевых/сырьевых цен без связи с расходами населения.
- Не ставь industrial_production_index для любой компании без производства/добычи/инфраструктуры.
- Не ставь mortality_rate для животных, метафор или &quot;смерти проекта&quot;.
- Не ставь family/demography только из-за слова &quot;дети&quot; или &quot;семья&quot; без социального события.

Каталог допустимых factor_id:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON с единственным верхним ключом &quot;items&quot;:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;mortality_rate&quot;,
          &quot;relevance&quot;: 0.3,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.45,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;погибшие -&gt; mortality_rate&quot;
        }
      ]
    }
  ]
}

Жёсткие требования:
- Верни все news_id из входа.
- Не используй другие верхние ключи, кроме &quot;items&quot;.
- Не возвращай factor_id вне каталога.
- Не возвращай факторы с relevance=0.
- Если факторов нет, используй &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence и reason до 12 слов.
- Никакого markdown и текста вне JSON.
</pre>

</details>

<details><summary><code>thinking_recall_latent_candidate_v3_think_true</code> · thinking recall · 5090 chars</summary>

<pre>Ты классифицируешь новости по социальным факторам. Режим: HIGH-RECALL CANDIDATES.

Это не финальный фильтр. Твоя задача - вернуть как можно больше вероятно связанных факторов,
чтобы downstream-фильтр потом отсёк слабые и ложные связи.

Главное отличие от обычной классификации:
- Не требуй прямого упоминания статистического показателя.
- Ищи event-level связь: событие в новости -&gt; социальное последствие -&gt; фактор.
- Если связь скрытая, но разумная, верни фактор с relevance=0.3 и confidence=0.35-0.55.
- Если связь явная, верни relevance=0.6.
- Если фактор главный смысл новости, верни relevance=0.9 или 1.0.
- Для релевантной новости обычно верни 1-5 факторов, для сложной новости можно до 8.
- Для нерелевантной новости верни пустой список.

Быстрая карта скрытых связей:
- Погиб, умер, погибшие, жертвы, летальный исход -&gt; mortality_rate.
- Раненые, тяжелые травмы, угроза жизни, здоровье пострадавших -&gt; life_expectancy.
- Преступление, нападение, обстрел, теракт, взрыв, мошенничество, коррупция, незаконные действия -&gt; crime_count.
- Госпитализация, скорая, больница, стационар, лечение пострадавших -&gt; hospitals.
- Поликлиника, амбулаторная помощь, первичное звено -&gt; outpatient_clinics.
- Врачи, медики, дефицит/работа медперсонала -&gt; qualified_doctors.
- Цены для населения, тарифы, инфляция, платежи, стоимость товаров/услуг -&gt; consumer_price_index.
- Доходы, зарплаты, пенсии, пособия, покупательная способность -&gt; per_capita_income или real_income_index.
- Прожиточный минимум, базовые расходы, минимальные выплаты -&gt; living_wage; если про детей -&gt; child_living_wage.
- Увольнения, найм, простой, занятость, рынок труда -&gt; unemployment_rate.
- Производство, добыча, заводы, энергетика, выпуск продукции, промышленная инфраструктура -&gt; industrial_production_index.
- Компании, открытие/закрытие бизнеса, банкротство, число организаций -&gt; enterprises_count.
- Жилье, расселение, аварийные дома, коммунальная доступность жилья -&gt; housing_area_per_capita.
- Новостройки/первичка/ипотека нового жилья -&gt; primary_housing_price_index.
- Вторичное жилье/готовое жилье -&gt; secondary_housing_price_index.
- Выбросы, дым, пожар с загрязнением воздуха, вредные вещества -&gt; air_pollution.
- Вода, стоки, водоснабжение, канализация, очистные сооружения -&gt; wastewater_discharge.
- Въезд людей, мигранты, туристы, беженцы, иностранные работники/студенты -&gt; international_inflow.
- Выезд людей, эмиграция, эвакуация за рубеж, отток граждан -&gt; international_outflow.
- Переезд/эвакуация/размещение внутри страны или региона -&gt; internal_arrivals/internal_departures.
- Рождение детей, рождаемость, демографические меры рождения -&gt; birth_rate_per_1000.
- Браки/свадьбы/регистрация брака -&gt; marriage_rate.
- Разводы/расторжение брака -&gt; divorce_rate.
- Аборт/ограничение или доступность абортов -&gt; abortions.
- Детские выплаты/пособия семьям с детьми -&gt; children_benefits.
- Маткапитал/сертификаты материнского капитала -&gt; maternity_capital.
- Пособие по уходу за ребёнком -&gt; childcare_allowance.
- Детсады/дошкольное образование/места для дошкольников -&gt; preschool_coverage.
- Жилье для многодетных семей -&gt; large_families_housing.
- Численность населения/депопуляция/массовое переселение -&gt; population_size.
- Младенцы умерли/угроза жизни младенцев -&gt; infant_mortality.
- Мужское/женское население только если есть демографический смысл -&gt; male_population/female_population.

Контроль ложных связей:
- Не ставь migration для экспорта, импорта, дипломатии и санкций без движения людей.
- Не ставь consumer_price_index для биржевых/сырьевых цен без связи с расходами населения.
- Не ставь industrial_production_index для любой компании без производства/добычи/инфраструктуры.
- Не ставь mortality_rate для животных, метафор или &quot;смерти проекта&quot;.
- Не ставь family/demography только из-за слова &quot;дети&quot; или &quot;семья&quot; без социального события.

Каталог допустимых factor_id:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON с единственным верхним ключом &quot;items&quot;:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;mortality_rate&quot;,
          &quot;relevance&quot;: 0.3,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.45,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;погибшие -&gt; mortality_rate&quot;
        }
      ]
    }
  ]
}

Жёсткие требования:
- Верни все news_id из входа.
- Не используй другие верхние ключи, кроме &quot;items&quot;.
- Не возвращай factor_id вне каталога.
- Не возвращай факторы с relevance=0.
- Если факторов нет, используй &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence и reason до 12 слов.
- Никакого markdown и текста вне JSON.
</pre>

</details>

<details><summary><code>thinking_recall_recall_max_v1_think_false</code> · thinking recall · 5598 chars</summary>

<pre>Проанализируй новости по социальным факторам.

Цель этого эксперимента: высокий recall. Лучше вернуть лишний слабый фактор с relevance=0.3, чем пропустить реальный фактор.
Дальше downstream-фильтр сможет отсечь слабые связи, поэтому не будь чрезмерно консервативным.

Важная политика разметки:
- Факторы здесь являются не только буквальными статистическими показателями, а event-level социальными сигналами.
- Прямой статистики в новости может не быть. Если событие очевидно относится к фактору, фактор нужно вернуть.
- Для одной новости можно вернуть несколько факторов. Обычно 1-4, максимум 6.
- Если фактор связан только очень косвенно, ставь relevance=0.3.
- Если фактор явно затронут, но не главный смысл новости, ставь relevance=0.6.
- Если фактор является главным смыслом новости, ставь relevance=0.9 или 1.0.
- Если нет ни одного фактора даже после event-level трактовки, верни пустой список factors.

Карта решений:
- crime_count: преступления, уголовные дела, атаки, обстрелы, теракты, насилие, убийства, ранения из-за атак, мошенничество, коррупция, незаконные действия, угрозы безопасности.
- mortality_rate: гибель людей, число погибших, смерть, рост жертв, летальные исходы, смертность. Единичная смерть тоже считается социальным сигналом.
- life_expectancy: здоровье людей, тяжелые травмы, угрозы жизни, массовые заболевания, безопасность жизни, состояние пострадавших, долгосрочные риски для здоровья.
- hospitals: больницы, госпитализация, скорая помощь, работа медорганизаций, доступность медицинской помощи, больничная инфраструктура.
- qualified_doctors: врачи, медицинский персонал, дефицит/наличие врачей, пострадавшие врачи или медики.
- industrial_production_index: производство, добыча, заводы, промышленность, энергетика, остановка/запуск производств, портовая/транспортная инфраструктура если она влияет на производство.
- enterprises_count: компании, бизнес-активность, банкротства, регистрации, закрытия, корпоративные решения, деятельность организаций.
- unemployment_rate: занятость, увольнения, наем, простой работников, корпоративные отпуска, рынок труда.
- consumer_price_index: инфляция, потребительские цены, тарифы, индексация платежей, рост/снижение стоимости товаров и услуг.
- per_capita_income, real_income_index, living_wage: доходы, зарплаты, пенсии, пособия, покупательная способность, рассрочка, долговая нагрузка населения.
- children_benefits, maternity_capital, childcare_allowance, child_living_wage: детские выплаты, маткапитал, поддержка семей с детьми, расходы на детей.
- housing_area_per_capita, primary_housing_price_index, secondary_housing_price_index, large_families_housing: жилье, жилищные условия, коммунальная доступность, цены на жилье, улучшение/ухудшение условий проживания.
- air_pollution: выбросы, загрязнение воздуха, пожары/взрывы/утечки с риском загрязнения воздуха.
- wastewater_discharge: загрязнение воды, сбросы, водоснабжение, очистные сооружения, канализация, коммунальная водная инфраструктура.
- international_inflow/outflow: приезд/выезд людей, миграция, беженцы, туристы, пересечение границ людьми. Не используй для обычной торговли, экспорта, дипломатии без движения людей.
- internal_arrivals/internal_departures: внутренняя миграция, переезд людей между регионами, эвакуация/возвращение внутри страны.
- population_size, birth_rate_per_1000, infant_mortality, male_population, female_population, marriage_rate, divorce_rate, abortions: демография, рождаемость, младенческая смертность, структура населения, браки/разводы, аборты.
- outpatient_clinics: поликлиники, амбулаторная помощь, первичное звено медицины.
- preschool_coverage: детские сады, дошкольное образование, доступность мест для детей.

Контроль ложных срабатываний:
- Не возвращай international_inflow/outflow для обычных внешнеполитических переговоров, экспорта, санкций или торговли, если нет движения людей.
- Не возвращай consumer_price_index для биржевых/сырьевых цен, если нет связи с потребительскими ценами, тарифами или расходами населения.
- Не возвращай industrial_production_index для любой компании автоматически; нужна связь с производством, выпуском, добычей, инфраструктурой или промышленной активностью.
- Не возвращай marriage_rate/family только из-за слова &quot;школьники&quot;, &quot;семья&quot; в нерелевантном контексте.

Оценки:
- sentiment: -1 сильный негатив, -0.5 умеренный негатив, 0 нейтрально, 0.5 умеренный позитив, 1 сильный позитив.
- pressure: социальное давление/риск от 0 до 1.
- confidence: уверенность от 0 до 1.
- label: positive, negative или neutral.

Факторы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON по схеме:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.6,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.8,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;до 12 слов&quot;
        }
      ]
    }
  ]
}

Требования:
- Верни все news_id из входа.
- В factors возвращай только factor_id из каталога.
- Не возвращай факторы с relevance=0.
- Если нет релевантных факторов, верни &quot;factors&quot;: [].
- Не добавляй текст вне JSON.
- evidence и reason должны быть короткими.
</pre>

</details>

<details><summary><code>thinking_recall_recall_max_v1_think_true</code> · thinking recall · 5598 chars</summary>

<pre>Проанализируй новости по социальным факторам.

Цель этого эксперимента: высокий recall. Лучше вернуть лишний слабый фактор с relevance=0.3, чем пропустить реальный фактор.
Дальше downstream-фильтр сможет отсечь слабые связи, поэтому не будь чрезмерно консервативным.

Важная политика разметки:
- Факторы здесь являются не только буквальными статистическими показателями, а event-level социальными сигналами.
- Прямой статистики в новости может не быть. Если событие очевидно относится к фактору, фактор нужно вернуть.
- Для одной новости можно вернуть несколько факторов. Обычно 1-4, максимум 6.
- Если фактор связан только очень косвенно, ставь relevance=0.3.
- Если фактор явно затронут, но не главный смысл новости, ставь relevance=0.6.
- Если фактор является главным смыслом новости, ставь relevance=0.9 или 1.0.
- Если нет ни одного фактора даже после event-level трактовки, верни пустой список factors.

Карта решений:
- crime_count: преступления, уголовные дела, атаки, обстрелы, теракты, насилие, убийства, ранения из-за атак, мошенничество, коррупция, незаконные действия, угрозы безопасности.
- mortality_rate: гибель людей, число погибших, смерть, рост жертв, летальные исходы, смертность. Единичная смерть тоже считается социальным сигналом.
- life_expectancy: здоровье людей, тяжелые травмы, угрозы жизни, массовые заболевания, безопасность жизни, состояние пострадавших, долгосрочные риски для здоровья.
- hospitals: больницы, госпитализация, скорая помощь, работа медорганизаций, доступность медицинской помощи, больничная инфраструктура.
- qualified_doctors: врачи, медицинский персонал, дефицит/наличие врачей, пострадавшие врачи или медики.
- industrial_production_index: производство, добыча, заводы, промышленность, энергетика, остановка/запуск производств, портовая/транспортная инфраструктура если она влияет на производство.
- enterprises_count: компании, бизнес-активность, банкротства, регистрации, закрытия, корпоративные решения, деятельность организаций.
- unemployment_rate: занятость, увольнения, наем, простой работников, корпоративные отпуска, рынок труда.
- consumer_price_index: инфляция, потребительские цены, тарифы, индексация платежей, рост/снижение стоимости товаров и услуг.
- per_capita_income, real_income_index, living_wage: доходы, зарплаты, пенсии, пособия, покупательная способность, рассрочка, долговая нагрузка населения.
- children_benefits, maternity_capital, childcare_allowance, child_living_wage: детские выплаты, маткапитал, поддержка семей с детьми, расходы на детей.
- housing_area_per_capita, primary_housing_price_index, secondary_housing_price_index, large_families_housing: жилье, жилищные условия, коммунальная доступность, цены на жилье, улучшение/ухудшение условий проживания.
- air_pollution: выбросы, загрязнение воздуха, пожары/взрывы/утечки с риском загрязнения воздуха.
- wastewater_discharge: загрязнение воды, сбросы, водоснабжение, очистные сооружения, канализация, коммунальная водная инфраструктура.
- international_inflow/outflow: приезд/выезд людей, миграция, беженцы, туристы, пересечение границ людьми. Не используй для обычной торговли, экспорта, дипломатии без движения людей.
- internal_arrivals/internal_departures: внутренняя миграция, переезд людей между регионами, эвакуация/возвращение внутри страны.
- population_size, birth_rate_per_1000, infant_mortality, male_population, female_population, marriage_rate, divorce_rate, abortions: демография, рождаемость, младенческая смертность, структура населения, браки/разводы, аборты.
- outpatient_clinics: поликлиники, амбулаторная помощь, первичное звено медицины.
- preschool_coverage: детские сады, дошкольное образование, доступность мест для детей.

Контроль ложных срабатываний:
- Не возвращай international_inflow/outflow для обычных внешнеполитических переговоров, экспорта, санкций или торговли, если нет движения людей.
- Не возвращай consumer_price_index для биржевых/сырьевых цен, если нет связи с потребительскими ценами, тарифами или расходами населения.
- Не возвращай industrial_production_index для любой компании автоматически; нужна связь с производством, выпуском, добычей, инфраструктурой или промышленной активностью.
- Не возвращай marriage_rate/family только из-за слова &quot;школьники&quot;, &quot;семья&quot; в нерелевантном контексте.

Оценки:
- sentiment: -1 сильный негатив, -0.5 умеренный негатив, 0 нейтрально, 0.5 умеренный позитив, 1 сильный позитив.
- pressure: социальное давление/риск от 0 до 1.
- confidence: уверенность от 0 до 1.
- label: positive, negative или neutral.

Факторы:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON по схеме:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.6,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.8,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;до 12 слов&quot;
        }
      ]
    }
  ]
}

Требования:
- Верни все news_id из входа.
- В factors возвращай только factor_id из каталога.
- Не возвращай факторы с relevance=0.
- Если нет релевантных факторов, верни &quot;factors&quot;: [].
- Не добавляй текст вне JSON.
- evidence и reason должны быть короткими.
</pre>

</details>

<details><summary><code>v3_social_signal_social_signal_v1</code> · v3 social signal · 5099 chars</summary>

<pre>Ты размечаешь новости для аналитической системы социальных рисков по логике ВКР.

Методическая рамка:
- новость = единица наблюдения;
- фактор = единица интерпретации;
- LLM-разметка = управляемая семантическая аннотация, а не автономная экспертная истина;
- новостной сигнал не равен фактическому состоянию социальной реальности;
- слабые и контекстные сигналы нужно сохранять, потому что из них в агрегации может собираться латентный паттерн.


Работай в режиме social-signal annotation.
Новость не является прямым измерением социальной реальности. Она является слабым цифровым сигналом, который потом агрегируется со статистикой и экспертной оценкой.
Поэтому нужно вернуть не один &quot;главный класс&quot;, а все факторы, для которых есть содержательная аналитическая связь.
Обычно у релевантной новости 1-5 факторов; у комплексной новости может быть больше.


Что считать факторной связью:
- Прямой сигнал: в тексте прямо описан фактор или его событие.
  Примеры: погибшие -&gt; mortality_rate; цены/тарифы -&gt; consumer_price_index; больница/скорая -&gt; hospitals.
- Контекстная содержательная связь: фактор не назван как статистика, но событие влияет на социальный контекст.
  Примеры: ранение врача -&gt; qualified_doctors и life_expectancy; повреждение домов -&gt; housing_area_per_capita.
- Слабая потенциальная связь: новость не про социальную статистику напрямую, но может быть микросигналом для аналитика.
  Примеры: санкции против промышленной компании -&gt; industrial_production_index и enterprises_count;
  Brent/Urals/топливо, если есть связь с расходами населения или инфляцией -&gt; consumer_price_index;
  меры поддержки семей -&gt; children_benefits / maternity_capital / childcare_allowance.

Не возвращай отдельное поле direct/context/weak. Используй эту шкалу только для relevance:
- relevance=0.9 или 1.0: фактор является главным или прямым смыслом новости.
- relevance=0.6: фактор явно затронут, но не единственный смысл.
- relevance=0.3: слабая, но реальная аналитическая связь, которую стоит сохранить для downstream-фильтра.
- Не возвращай фактор, если связи нет даже как слабого социального сигнала.

Важные co-label правила:
- Если событие связано с ценами/инфляцией/тарифами, часто также проверь real_income_index и per_capita_income.
- Если событие связано с промышленностью, добычей, энергетикой, портами, заводами, логистикой производства, проверь industrial_production_index и enterprises_count.
- Если есть атака, преступление, обстрел, теракт, мошенничество или коррупция, проверь crime_count; если есть погибшие, добавь mortality_rate; если есть раненые/угроза жизни, добавь life_expectancy; если есть больница/скорая, добавь hospitals.
- Если есть жилье, аварийные дома, повреждение домов, расселение или коммунальная инфраструктура, проверь housing_area_per_capita; если речь о ценах жилья, проверь primary_housing_price_index / secondary_housing_price_index.
- Если есть врачи, медики, дефицит кадров или пострадавшие медработники, проверь qualified_doctors.
- Migration factors используй только когда есть движение людей: въезд, выезд, эвакуация, туристы, мигранты, беженцы, переселение. Не используй для обычной дипломатии, экспорта, санкций и торговли без движения людей.

Контроль ложных связей:
- Не ставь consumer_price_index для любых биржевых цен автоматически. Нужна связь с расходами населения, тарифами, потребительскими товарами или инфляцией.
- Не ставь industrial_production_index для любой компании автоматически. Нужна связь с производством, добычей, выпуском, инфраструктурой или промышленной активностью.
- Не ставь family/demography только из-за слов &quot;дети&quot; или &quot;семья&quot;, если нет демографического, социального или поддерживающего события.
- Не ставь mortality_rate для смерти животных, метафор или &quot;смерти проекта&quot;.

Примеры стиля v3-разметки из gold dataset:
[{&quot;news&quot;:&quot;{EXAMPLE_NEWS_TITLE}&quot;,&quot;expected_factors&quot;:[&quot;{FACTOR_ID}&quot;],&quot;logic&quot;:&quot;{EXAMPLE_LOGIC}&quot;}]

Каталог factor_id:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON с единственным верхним ключом &quot;items&quot;:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;mortality_rate&quot;,
          &quot;relevance&quot;: 0.6,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.75,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;погибшие -&gt; mortality_rate&quot;
        }
      ]
    }
  ]
}

Жесткие требования:
- Верни все news_id из входа.
- В factors возвращай только factor_id из каталога.
- Не возвращай relevance=0.
- Если совсем нет социальной связи, используй &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence и reason до 12 слов.
- Никакого markdown и текста вне JSON.
</pre>

</details>

<details><summary><code>v3_social_signal_social_signal_v2</code> · v3 social signal · 5116 chars</summary>

<pre>Ты размечаешь новости для аналитической системы социальных рисков по логике ВКР.

Методическая рамка:
- новость = единица наблюдения;
- фактор = единица интерпретации;
- LLM-разметка = управляемая семантическая аннотация, а не автономная экспертная истина;
- новостной сигнал не равен фактическому состоянию социальной реальности;
- слабые и контекстные сигналы нужно сохранять, потому что из них в агрегации может собираться латентный паттерн.


Работай в режиме расширенного аналитического recall.
Не останавливайся после первого очевидного фактора: сделай мысленный checklist по всем 36 factor_id.
Верни все содержательные связи, если новость может быть использована аналитиком как слабый социальный сигнал.
Обычно у релевантной новости 2-5 факторов; у комплексной экономической, промышленной, военной, медицинской или социальной новости может быть до 8-10 факторов.


Что считать факторной связью:
- Прямой сигнал: в тексте прямо описан фактор или его событие.
  Примеры: погибшие -&gt; mortality_rate; цены/тарифы -&gt; consumer_price_index; больница/скорая -&gt; hospitals.
- Контекстная содержательная связь: фактор не назван как статистика, но событие влияет на социальный контекст.
  Примеры: ранение врача -&gt; qualified_doctors и life_expectancy; повреждение домов -&gt; housing_area_per_capita.
- Слабая потенциальная связь: новость не про социальную статистику напрямую, но может быть микросигналом для аналитика.
  Примеры: санкции против промышленной компании -&gt; industrial_production_index и enterprises_count;
  Brent/Urals/топливо, если есть связь с расходами населения или инфляцией -&gt; consumer_price_index;
  меры поддержки семей -&gt; children_benefits / maternity_capital / childcare_allowance.

Не возвращай отдельное поле direct/context/weak. Используй эту шкалу только для relevance:
- relevance=0.9 или 1.0: фактор является главным или прямым смыслом новости.
- relevance=0.6: фактор явно затронут, но не единственный смысл.
- relevance=0.3: слабая, но реальная аналитическая связь, которую стоит сохранить для downstream-фильтра.
- Не возвращай фактор, если связи нет даже как слабого социального сигнала.

Важные co-label правила:
- Если событие связано с ценами/инфляцией/тарифами, часто также проверь real_income_index и per_capita_income.
- Если событие связано с промышленностью, добычей, энергетикой, портами, заводами, логистикой производства, проверь industrial_production_index и enterprises_count.
- Если есть атака, преступление, обстрел, теракт, мошенничество или коррупция, проверь crime_count; если есть погибшие, добавь mortality_rate; если есть раненые/угроза жизни, добавь life_expectancy; если есть больница/скорая, добавь hospitals.
- Если есть жилье, аварийные дома, повреждение домов, расселение или коммунальная инфраструктура, проверь housing_area_per_capita; если речь о ценах жилья, проверь primary_housing_price_index / secondary_housing_price_index.
- Если есть врачи, медики, дефицит кадров или пострадавшие медработники, проверь qualified_doctors.
- Migration factors используй только когда есть движение людей: въезд, выезд, эвакуация, туристы, мигранты, беженцы, переселение. Не используй для обычной дипломатии, экспорта, санкций и торговли без движения людей.

Контроль ложных связей:
- Не ставь consumer_price_index для любых биржевых цен автоматически. Нужна связь с расходами населения, тарифами, потребительскими товарами или инфляцией.
- Не ставь industrial_production_index для любой компании автоматически. Нужна связь с производством, добычей, выпуском, инфраструктурой или промышленной активностью.
- Не ставь family/demography только из-за слов &quot;дети&quot; или &quot;семья&quot;, если нет демографического, социального или поддерживающего события.
- Не ставь mortality_rate для смерти животных, метафор или &quot;смерти проекта&quot;.

Примеры стиля v3-разметки из gold dataset:
[{&quot;news&quot;:&quot;{EXAMPLE_NEWS_TITLE}&quot;,&quot;expected_factors&quot;:[&quot;{FACTOR_ID}&quot;],&quot;logic&quot;:&quot;{EXAMPLE_LOGIC}&quot;}]

Каталог factor_id:
[{&quot;factor_id&quot;:&quot;{FACTOR_ID}&quot;,&quot;dashboard_name&quot;:&quot;{DASHBOARD_NAME}&quot;,&quot;social_signal_meaning&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;count_as_signal_when&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;do_not_use_when&quot;:&quot;{DO_NOT_USE_WHEN}&quot;,&quot;positive_signal&quot;:&quot;{POSITIVE_SIGNAL}&quot;,&quot;negative_signal&quot;:&quot;{NEGATIVE_SIGNAL}&quot;}]

Новости:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни строго JSON с единственным верхним ключом &quot;items&quot;:
{
  &quot;items&quot;: [
    {
      &quot;news_id&quot;: &quot;string&quot;,
      &quot;factors&quot;: [
        {
          &quot;factor_id&quot;: &quot;mortality_rate&quot;,
          &quot;relevance&quot;: 0.6,
          &quot;sentiment&quot;: -0.5,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.75,
          &quot;label&quot;: &quot;negative&quot;,
          &quot;evidence&quot;: &quot;короткая фраза из новости&quot;,
          &quot;reason&quot;: &quot;погибшие -&gt; mortality_rate&quot;
        }
      ]
    }
  ]
}

Жесткие требования:
- Верни все news_id из входа.
- В factors возвращай только factor_id из каталога.
- Не возвращай relevance=0.
- Если совсем нет социальной связи, используй &quot;factors&quot;: [].
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence и reason до 12 слов.
- Никакого markdown и текста вне JSON.
</pre>

</details>

<details><summary><code>v3_social_signal_social_signal_v3</code> · v3 social signal · 3231 chars</summary>

<pre>Разметь ОДНУ новость как источник слабых социальных сигналов.

Цель: вернуть ВСЕ factor_id, с которыми есть содержательная связь. Не выбирай один главный класс.
Новость - единица наблюдения, фактор - единица интерпретации. Слабый сигнал сохраняем, если аналитик мог бы использовать его в агрегации.


Если релевантная новость дала только 0-1 фактор, перепроверь co-labels.


Шкала relevance:
0.9 = прямой/главный сигнал.
0.6 = явная содержательная связь.
0.3 = слабая, но реальная аналитическая связь.
Не возвращай фактор без связи.

Перед ответом молча проверь группы: безопасность/смертность, медицина, цены/доходы, производство/компании, жилье/ЖКХ, экология, семья/дети, миграция/демография.

Co-label правила:
- цены/тарифы/инфляция -&gt; consumer_price_index; часто также real_income_index/per_capita_income.
- промышленность/добыча/энергетика/заводы/порты/производственная логистика -&gt; industrial_production_index; часто enterprises_count.
- преступление/атака/обстрел/теракт/коррупция/мошенничество -&gt; crime_count.
- погибшие -&gt; mortality_rate; раненые/угроза жизни -&gt; life_expectancy; больница/скорая/госпитализация -&gt; hospitals.
- повреждение домов/нет воды/нет света/расселение/ЖКХ -&gt; housing_area_per_capita; вода/стоки/водоснабжение -&gt; wastewater_discharge.
- врачи/медики/дефицит кадров/пострадавший врач -&gt; qualified_doctors.
- миграционные факторы только при движении людей: въезд, выезд, эвакуация, туристы, мигранты, беженцы, переселение. Не для дипломатии/экспорта/санкций.

Примеры:
[{&quot;news&quot;:&quot;В результате удара по энергообъекту есть погибший и раненые&quot;,&quot;expected&quot;:[&quot;crime_count&quot;,&quot;mortality_rate&quot;,&quot;life_expectancy&quot;,&quot;industrial_production_index&quot;],&quot;logic&quot;:&quot;атака/удар, погибший, раненые, энергообъект&quot;},{&quot;news&quot;:&quot;Авария на водоканале оставила без воды многоэтажные дома&quot;,&quot;expected&quot;:[&quot;housing_area_per_capita&quot;,&quot;wastewater_discharge&quot;],&quot;logic&quot;:&quot;условия проживания + водная/коммунальная инфраструктура&quot;},{&quot;news&quot;:&quot;Утвержден бюджет ФОМС и Социального фонда&quot;,&quot;expected&quot;:[&quot;hospitals&quot;,&quot;per_capita_income&quot;,&quot;living_wage&quot;,&quot;children_benefits&quot;],&quot;logic&quot;:&quot;медицина, выплаты, доходы/минимальное обеспечение&quot;},{&quot;news&quot;:&quot;Санкции затронули энергетическую компанию&quot;,&quot;expected&quot;:[&quot;industrial_production_index&quot;,&quot;enterprises_count&quot;],&quot;logic&quot;:&quot;промышленность и деятельность компании; consumer_price_index только если есть цены/тарифы для населения&quot;},{&quot;news&quot;:&quot;Переговоры РФ-США без перемещения людей&quot;,&quot;expected&quot;:[],&quot;logic&quot;:&quot;не ставить international_inflow/outflow только за дипломатию&quot;}]

Каталог:
[{&quot;id&quot;:&quot;{FACTOR_ID}&quot;,&quot;signal&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;use&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;avoid&quot;:&quot;{DO_NOT_USE_WHEN}&quot;}]

Новость:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни только JSON:
{&quot;items&quot;:[{&quot;news_id&quot;:&quot;string&quot;,&quot;factors&quot;:[{&quot;factor_id&quot;:&quot;crime_count&quot;,&quot;relevance&quot;:0.6,&quot;sentiment&quot;:-0.5,&quot;pressure&quot;:0.5,&quot;confidence&quot;:0.8,&quot;label&quot;:&quot;negative&quot;,&quot;evidence&quot;:&quot;фраза из новости&quot;,&quot;reason&quot;:&quot;атака -&gt; crime_count&quot;}]}]}

Требования:
- Верни ровно один item на входной news_id.
- factors может содержать 0..10 факторов.
- factor_id только из каталога.
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence и reason до 10 слов.
- Никакого текста вне JSON.
</pre>

</details>

<details><summary><code>v3_social_signal_social_signal_v4</code> · v3 social signal · 3904 chars</summary>

<pre>Разметь ОДНУ новость как источник слабых социальных сигналов.

Цель: вернуть ВСЕ factor_id, с которыми есть содержательная связь. Не выбирай один главный класс.
Новость - единица наблюдения, фактор - единица интерпретации. Слабый сигнал сохраняем, если аналитик мог бы использовать его в агрегации.


Количественный prior:
- В v3-разметке релевантная новость обычно содержит около 3 релевантных факторов.
- Цель не минимальный список, а полный аналитический набор связей.
- Если новость явно релевантна и ты вернул только 1 фактор, это подозрительно: перепроверь все co-label правила.
- Для обычной релевантной новости целевой диапазон 2-4 фактора.
- Для комплексной новости про атаку, аварию, бюджет, санкции, промышленность, цены, медицину или ЖКХ допустимо 4-8 факторов.
- Лучше вернуть дополнительную потенциальную связь с relevance=0.3, чем пропустить фактор, который потом может быть отфильтрован downstream-верификатором.
- Но не возвращай бессодержательные ассоциации: у каждого factor_id должна быть короткая причинная цепочка в reason.


Шкала relevance:
0.9 = прямой/главный сигнал.
0.6 = явная содержательная связь.
0.3 = слабая, но реальная аналитическая связь.
Не возвращай фактор без связи.

Перед ответом молча проверь группы: безопасность/смертность, медицина, цены/доходы, производство/компании, жилье/ЖКХ, экология, семья/дети, миграция/демография.

Co-label правила:
- цены/тарифы/инфляция -&gt; consumer_price_index; часто также real_income_index/per_capita_income.
- промышленность/добыча/энергетика/заводы/порты/производственная логистика -&gt; industrial_production_index; часто enterprises_count.
- преступление/атака/обстрел/теракт/коррупция/мошенничество -&gt; crime_count.
- погибшие -&gt; mortality_rate; раненые/угроза жизни -&gt; life_expectancy; больница/скорая/госпитализация -&gt; hospitals.
- повреждение домов/нет воды/нет света/расселение/ЖКХ -&gt; housing_area_per_capita; вода/стоки/водоснабжение -&gt; wastewater_discharge.
- врачи/медики/дефицит кадров/пострадавший врач -&gt; qualified_doctors.
- миграционные факторы только при движении людей: въезд, выезд, эвакуация, туристы, мигранты, беженцы, переселение. Не для дипломатии/экспорта/санкций.

Примеры:
[{&quot;news&quot;:&quot;В результате удара по энергообъекту есть погибший и раненые&quot;,&quot;expected&quot;:[&quot;crime_count&quot;,&quot;mortality_rate&quot;,&quot;life_expectancy&quot;,&quot;industrial_production_index&quot;],&quot;logic&quot;:&quot;атака/удар, погибший, раненые, энергообъект&quot;},{&quot;news&quot;:&quot;Авария на водоканале оставила без воды многоэтажные дома&quot;,&quot;expected&quot;:[&quot;housing_area_per_capita&quot;,&quot;wastewater_discharge&quot;],&quot;logic&quot;:&quot;условия проживания + водная/коммунальная инфраструктура&quot;},{&quot;news&quot;:&quot;Утвержден бюджет ФОМС и Социального фонда&quot;,&quot;expected&quot;:[&quot;hospitals&quot;,&quot;per_capita_income&quot;,&quot;living_wage&quot;,&quot;children_benefits&quot;],&quot;logic&quot;:&quot;медицина, выплаты, доходы/минимальное обеспечение&quot;},{&quot;news&quot;:&quot;Санкции затронули энергетическую компанию&quot;,&quot;expected&quot;:[&quot;industrial_production_index&quot;,&quot;enterprises_count&quot;],&quot;logic&quot;:&quot;промышленность и деятельность компании; consumer_price_index только если есть цены/тарифы для населения&quot;},{&quot;news&quot;:&quot;Переговоры РФ-США без перемещения людей&quot;,&quot;expected&quot;:[],&quot;logic&quot;:&quot;не ставить international_inflow/outflow только за дипломатию&quot;}]

Каталог:
[{&quot;id&quot;:&quot;{FACTOR_ID}&quot;,&quot;signal&quot;:&quot;{SOCIAL_SIGNAL_MEANING}&quot;,&quot;use&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;,&quot;avoid&quot;:&quot;{DO_NOT_USE_WHEN}&quot;}]

Новость:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни только JSON:
{&quot;items&quot;:[{&quot;news_id&quot;:&quot;string&quot;,&quot;factors&quot;:[{&quot;factor_id&quot;:&quot;crime_count&quot;,&quot;relevance&quot;:0.6,&quot;sentiment&quot;:-0.5,&quot;pressure&quot;:0.5,&quot;confidence&quot;:0.8,&quot;label&quot;:&quot;negative&quot;,&quot;evidence&quot;:&quot;фраза из новости&quot;,&quot;reason&quot;:&quot;атака -&gt; crime_count&quot;}]}]}

Требования:
- Верни ровно один item на входной news_id.
- factors может содержать 0..10 факторов.
- factor_id только из каталога.
- sentiment: -1, -0.5, 0, 0.5, 1.
- label: positive, negative или neutral.
- evidence и reason до 10 слов.
- Никакого текста вне JSON.
</pre>

</details>

<details><summary><code>v3_social_signal_social_signal_v5</code> · v3 social signal · 1957 chars</summary>

<pre>Разметь одну новость для мониторинга социальных рисков.

Задача: найти ВСЕ содержательные связи news -&gt; factor_id. Это не выбор одной рубрики.

Жесткий prior по полноте:
- В v3 gold релевантная новость обычно имеет около 3 факторов.
- Нормальный диапазон: 2-4 фактора.
- 1 фактор допустим только если новость очень узкая.
- Для комплексных тем допустимо 4-8 факторов.
- Лучше добавить потенциальную связь с relevance=0.3, чем пропустить co-label.
- Но каждый фактор должен иметь короткую причинную цепочку в reason.

relevance:
0.9 прямой/главный сигнал; 0.6 явная связь; 0.3 слабая потенциальная связь.

Быстрый checklist:
- атака/преступление -&gt; crime_count; погибшие -&gt; mortality_rate; раненые -&gt; life_expectancy; больница/скорая -&gt; hospitals.
- цены/тарифы/инфляция -&gt; consumer_price_index; часто также real_income_index/per_capita_income.
- производство/добыча/энергетика/завод/порт/логистика -&gt; industrial_production_index; часто enterprises_count.
- компания/банкротство/открытие/закрытие/бизнес -&gt; enterprises_count.
- дома/ЖКХ/нет воды/нет света/расселение -&gt; housing_area_per_capita; вода/стоки -&gt; wastewater_discharge.
- врачи/медики/медперсонал -&gt; qualified_doctors.
- выплаты семьям/детям -&gt; children_benefits/living_wage/per_capita_income.
- миграция только при движении людей: въезд/выезд/эвакуация/туристы/беженцы/переселение. Не для дипломатии/экспорта/санкций.

Каталог:
[{&quot;id&quot;:&quot;{FACTOR_ID}&quot;,&quot;hint&quot;:&quot;{COUNT_AS_SIGNAL_WHEN}&quot;}]

Новость:
[{&quot;news_id&quot;:&quot;{NEWS_ID}&quot;,&quot;date&quot;:&quot;{DATE}&quot;,&quot;title&quot;:&quot;{TITLE}&quot;,&quot;summary&quot;:&quot;{SUMMARY}&quot;,&quot;text&quot;:&quot;{NEWS_TEXT}&quot;,&quot;url&quot;:&quot;{URL}&quot;}]

Верни только JSON:
{&quot;items&quot;:[{&quot;news_id&quot;:&quot;string&quot;,&quot;factors&quot;:[{&quot;factor_id&quot;:&quot;crime_count&quot;,&quot;relevance&quot;:0.6,&quot;sentiment&quot;:-0.5,&quot;pressure&quot;:0.5,&quot;confidence&quot;:0.8,&quot;label&quot;:&quot;negative&quot;,&quot;evidence&quot;:&quot;фраза&quot;,&quot;reason&quot;:&quot;атака -&gt; crime_count&quot;}]}]}

Требования: factor_id только из каталога; factors 0..8; sentiment -1/-0.5/0/0.5/1; label positive/neutral/negative; evidence/reason до 8 слов; без markdown.
</pre>

</details>

<details><summary><code>broad_balanced</code> · v4 direct search · 2704 chars</summary>

<pre># broad_balanced

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Broad all-signal target:
- Return direct, context, and weak but explainable social signals.
- Keep weak signals only when evidence in the text gives a plausible analytical channel.
- Typical relevant news has around 3 factors; simple news can have 1; complex news can have 4-8.
- Use relevance 0.80-1.00 direct, 0.55-0.79 context, 0.30-0.54 weak.


Hard negative rules:
- Do not mark international_inflow/outflow for diplomacy, sanctions, exports/imports, negotiations, documents, ships, logistics, or trade unless people physically move.
- Do not mark consumer_price_index for stock prices, IPO, company revenue, commodity quotes, or exchange news without a channel to consumer costs/tariffs/inflation.
- Do not mark industrial_production_index for a generic company, bank, investment, or financial transaction without production/extraction/infrastructure/output.
- Do not mark hospitals for water/electricity/utility accidents unless medical care, ambulance, hospitalization, or medical institutions are mentioned.
- Do not mark air_pollution for weather, temperature, rain, cold, or forecasts without pollution/smoke/emissions.
- Do not mark mortality_rate when the text explicitly says there are no casualties.
- Do not mark child/family factors from the word &quot;family&quot; alone.

Balance recall and precision; target 2-4 factors for ordinary relevant news.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>broad_recall</code> · v4 direct search · 1840 chars</summary>

<pre># broad_recall

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Broad all-signal target:
- Return direct, context, and weak but explainable social signals.
- Keep weak signals only when evidence in the text gives a plausible analytical channel.
- Typical relevant news has around 3 factors; simple news can have 1; complex news can have 4-8.
- Use relevance 0.80-1.00 direct, 0.55-0.79 context, 0.30-0.54 weak.

High recall: prefer adding a plausible weak/context factor with relevance=0.30 rather than missing an analytical signal. Still obey hard negatives.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>broad_weak_context</code> · v4 direct search · 1822 chars</summary>

<pre># broad_weak_context

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Broad all-signal target:
- Return direct, context, and weak but explainable social signals.
- Keep weak signals only when evidence in the text gives a plausible analytical channel.
- Typical relevant news has around 3 factors; simple news can have 1; complex news can have 4-8.
- Use relevance 0.80-1.00 direct, 0.55-0.79 context, 0.30-0.54 weak.

Focus on context and weak signals too. Direct signals should still receive higher relevance. Explain the channel in reason.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_brief_thinking</code> · v4 direct search · 3157 chars</summary>

<pre># direct_brief_thinking

Task: annotate Russian news with direct social-signal factors.

Think very briefly. Do not do long chain-of-thought. Decide factors, then output JSON only.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Return a factor only if the news itself contains a direct event/signal for it.
No broad geopolitics, no long causal chains, no generic context.
Most relevant news has 1-3 direct factors; accidents/attacks can have 3-5.

Mapping rules:
- killed, dead, fatality, death toll -&gt; mortality_rate
- birth, newborn, fertility -&gt; birth_rate_per_1000
- infant death -&gt; infant_mortality and mortality_rate
- injured, disease, life/health threat -&gt; life_expectancy
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals
- clinic, outpatient visit/polyclinic -&gt; outpatient_clinics
- doctors, medical staff, physician shortage -&gt; qualified_doctors
- crime, fraud, corruption, attack, violence, criminal case -&gt; crime_count
- price, inflation, tariff, fuel/food/utility cost -&gt; consumer_price_index
- income, wage, salary, pension, household payment -&gt; per_capita_income
- purchasing power, debt burden, income eroded by inflation -&gt; real_income_index
- unemployment, layoffs, job loss -&gt; unemployment_rate
- plant, production, extraction, factory, industrial output, energy generation -&gt; industrial_production_index
- company, bank, business, enterprise, bankruptcy, opening/closure -&gt; enterprises_count
- housing damage, resettlement, home conditions, housing utilities -&gt; housing_area_per_capita
- new-build housing prices -&gt; primary_housing_price_index
- resale housing prices -&gt; secondary_housing_price_index
- air emissions, smoke, air pollution -&gt; air_pollution
- sewage, wastewater, polluted water discharge -&gt; wastewater_discharge
- migration, evacuation, arrivals/departures of people -&gt; corresponding migration factor

Use relevance:
- 0.85-1.00 main direct signal
- 0.60-0.84 secondary direct signal
- never below 0.60 in this direct prompt

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no direct factor exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_business_migration_fixed</code> · v4 direct search · 3770 chars</summary>

<pre># direct_business_migration_fixed

Task: annotate Russian news titles/texts with DIRECT social-signal factors.

Think briefly, then output JSON only. No long reasoning outside JSON.

Important: the title is enough evidence. If the title directly names a company, bank, plant, project, export, bankruptcy, profit, investment quota, production plan, merger/liquidation, lawsuit against a company, or business activity, mark `enterprises_count`. This dataset treats corporate/economic actors as direct enterprise signals.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct mapping:
- company, bank, business, enterprise, corporation, plant as legal/economic actor, joint venture, export contract, bankruptcy, liquidation, corporate profit/loss, lawsuit against company, production abroad, investment quota -&gt; enterprises_count
- factory, extraction, gas/oil/LNG/coal, power generation, industrial output, production volume, plant operation -&gt; industrial_production_index
- consumer price, inflation, tariff, fuel/food/utility price -&gt; consumer_price_index
- crime, attack, shelling, fraud, corruption, criminal case, detention, violence -&gt; crime_count
- killed, dead, fatality, death toll -&gt; mortality_rate
- injured, disease, life/health threat -&gt; life_expectancy
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals
- doctor, physician, medical worker -&gt; qualified_doctors
- housing damage, home repair/resettlement, living space, housing utilities -&gt; housing_area_per_capita
- air emissions, smoke, air pollution -&gt; air_pollution
- sewage, wastewater, polluted water discharge -&gt; wastewater_discharge
- birth, newborn, fertility -&gt; birth_rate_per_1000
- infant death -&gt; infant_mortality and mortality_rate
- unemployment, layoffs, job loss -&gt; unemployment_rate
- wage, salary, pension, household payment -&gt; per_capita_income
- purchasing power, income eroded by inflation, debt burden -&gt; real_income_index

Strict migration rule:
- Use international_inflow/outflow only for physical people migrating or entering/leaving a country.
- Use internal_arrivals/departures only for physical people moving/evacuating inside the country.
- Do NOT use migration factors for exports/imports, foreign companies, sanctions, diplomacy, ships, aircraft restrictions, money flows, markets, investments, or international cooperation.

Relevance:
- 0.85-1.00 main direct signal.
- 0.60-0.84 secondary direct signal.
- Do not use relevance below 0.60.
- Typical relevant news has 1-3 direct factors; complex attacks/accidents/corporate-industrial news can have 3-5.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;enterprises_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;neutral&quot;,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;company/business event -&gt; enterprises_count&quot;
        }
      ]
    }
  ]
}

If no direct factor exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_checklist</code> · v4 direct search · 3130 chars</summary>

<pre># direct_checklist

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.

Before JSON, silently check safety/death/health/prices/income/jobs/production/business/housing/utilities/ecology/family/migration. Return only JSON.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_core_balanced_v8</code> · v4 direct search · 7357 chars</summary>

<pre># direct_core_balanced_v8

Task: annotate Russian news titles/texts with DIRECT social-signal factors.

Think briefly, then output JSON only. No prose outside JSON.

Goal: high direct micro-F1 on the v4 gold dataset. Keep the v7 gains on prices/medicine, but recover business/enterprise recall and avoid over-broad industrial/mortality labels.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Core direct factors:

1. enterprises_count
- Mark for any named company, bank, business group, commercial operator, plant as business actor, export/import contract, corporate production plan, corporate investment, bankruptcy, liquidation, profit/loss, dividend, corporate lawsuit, sanctions against a company, state regulation of business activity.
- Include foreign companies when the news has Russian/economic relevance, energy, trade, sanctions, logistics, or market effect.
- Usually mark enterprises_count together with industrial_production_index when a company produces, extracts, transports, imports, exports, or supplies industrial goods, energy, aircraft, vehicles, equipment, fuel, food, raw materials.
- Do not mark enterprises_count for a purely government/cultural/legal story unless firms, markets, operators, or commercial activity are visible.

2. industrial_production_index
- Mark for plant/factory, extraction, oil/gas/coal/uranium/LNG, electricity and power infrastructure, production volumes, mining, refinery, industrial goods, aircraft/vehicles/equipment, energy, fuel, industrial exports/imports, ports, tankers, railways, airports, logistics shutdowns, production launch/stop, industrial accidents.
- Mark airport/rail/port shutdowns as industrial/logistics activity.
- Do not mark industrial_production_index for pure stock-market movement, pure finance, generic sanctions, courts, culture, or politics unless production, extraction, energy, commodity, infrastructure, transport/logistics, or supply chain is visible.

3. consumer_price_index
- Mark for explicit prices, inflation, tariffs, fuel/food/utility prices, central-bank rate, import duty, oil/gas/coal price, exchange rate, food markets.
- Also mark visible supply/logistics/production shocks if they plausibly affect consumer prices of food, fuel, energy, utilities, transport, imported goods, or consumer services.
- Do not mark symbolic/cultural/political news as CPI unless there is a visible price, rate, tariff, commodity, supply, production, logistics, trade, or consumer-cost channel.

4. crime_count
- Mark for crime, court/criminal case, criminal punishment, detention/arrest, fraud, corruption, law enforcement, prosecutor/FSB/police, prison/convicts, attack, shelling, drone strike, sabotage, terrorism, military/security incident, strategic-facility disruption.
- Also mark genocide/crimes memory or legal definition of crimes.
- Do not mark ordinary economic, stock-market, health, ecology, or transport news unless the text has security/law-enforcement/attack/operational-risk framing.

5. mortality_rate
- Mark for killed/dead/fatalities/death toll.
- Mark for genocide/death memory, war casualties/losses, severe fire/explosion/attack, fatal accident, or death-risk disaster when death/fatal hazard is central.
- Do not mark routine market/economic news as mortality_rate unless it explicitly references deaths, war losses, genocide, fatality, or severe disaster.

Medical/ecology/housing:
- life_expectancy: mass illness, poisoning, infection, injuries, health threat, dangerous event affecting health.
- hospitals: hospital, ambulance, hospitalization, emergency medical care, people brought/transferred to doctors; if a wounded person is in hospital, mark hospitals.
- qualified_doctors: doctors, physicians, medical staff, ambulance doctor.
- air_pollution: smoke, emissions, oil spill/fire, fuel contamination, dirty air, environmental contamination of coast/air.
- wastewater_discharge: sewage, wastewater, polluted water discharge, water utility accident.
- housing_area_per_capita: damaged homes, housing repair/resettlement, living space, housing utilities, property/housing transaction rules.

Income/labor/demography:
- unemployment_rate: layoffs, job loss, labor bans/restrictions, unemployment.
- per_capita_income: wages, pensions, payments, taxes, compensation, household money.
- real_income_index: purchasing power or income explicitly linked to prices/inflation/debt burden.
- family/demographic factors require literal mention of births, marriages, divorces, children benefits, maternity capital, abortions, infant deaths, population size.

Strict migration rule:
- international_inflow/outflow only for physical people entering/leaving a country: migrants, tourists/visas, refugees, prisoners crossing border.
- internal_arrivals/departures only for physical people moving/evacuating inside the country.
- Never use migration for exports/imports, foreign companies, sanctions, diplomacy, ships, aircraft restrictions, money flows, markets, investments, or generic international cooperation.

Calibrating examples:
- &quot;Старт вечерней сессии на срочном рынке Мосбиржи задерживается&quot; -&gt; crime_count.
- &quot;Аэропорты Волгограда и Краснодара приостановили работу&quot; -&gt; industrial_production_index.
- &quot;Venture Global получил разрешение на допэкспорт СПГ&quot; -&gt; enterprises_count, industrial_production_index.
- &quot;Российские ледоколы помогают финским судам&quot; -&gt; enterprises_count and industrial_production_index if the body discusses cargo/transport/navigation service.
- &quot;Запросы из-за рубежа на поставки Ил-76&quot; -&gt; enterprises_count, industrial_production_index, consumer_price_index.
- &quot;Мировые цены на продовольствие выросли&quot; -&gt; consumer_price_index, industrial_production_index.
- &quot;Более 100 человек обратились в больницу с кишечной инфекцией&quot; -&gt; life_expectancy, hospitals.
- &quot;Уголовное наказание за отрицание геноцида&quot; -&gt; crime_count, mortality_rate.

Relevance:
- 0.90-0.95 for main direct signal.
- 0.75-0.89 for secondary direct signal.
- 0.60-0.74 for visible but weaker direct signal.
- Do not use relevance below 0.60.
- Typical relevant news has 1-3 direct factors; complex industrial/security/business/price news can have 3-5.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;enterprises_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;neutral&quot;,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;business actor / enterprise activity -&gt; enterprises_count&quot;
        }
      ]
    }
  ]
}

If no direct factor exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_core_clean_fewshot_v6</code> · v4 direct search · 6776 chars</summary>

<pre># direct_core_clean_fewshot_v6

Task: annotate Russian news with DIRECT social-signal factors for the current v4 gold dataset.

Think briefly, then output JSON only. No prose outside JSON.

Important calibration:
- In this dataset `direct` often means &quot;direct analytical signal&quot;, not a literal Rosstat measurement.
- Do not be too conservative on the main economic/security factors. A news item usually has 1-3 direct factors.
- Return only factors that have a visible text channel in the title or body. Do not invent hidden geopolitics.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Gold-style examples to imitate:
- &quot;Старт вечерней сессии на срочном рынке Мосбиржи задерживается&quot; -&gt; crime_count. Operational disruption in a strategic market is a security/operational-risk signal.
- &quot;Аэропорты Волгограда и Краснодара приостановили работу&quot; -&gt; industrial_production_index. Airport/transport shutdown affects industrial and logistics activity.
- &quot;В посольстве РФ рассказали о планах Британии отказаться от российского урана к 2028 году&quot; -&gt; industrial_production_index, enterprises_count, air_pollution. Uranium/energy commodity and business/export channel.
- &quot;Глава Минпромторга сообщил о запросах из-за рубежа на поставки Ил-76&quot; -&gt; industrial_production_index, enterprises_count, consumer_price_index. Industrial supply/export demand can be a price-pressure channel.
- &quot;Мировые цены на продовольствие выросли&quot; -&gt; consumer_price_index and industrial_production_index.
- &quot;Путин поручил разработать план подъема затонувших танкеров&quot; -&gt; air_pollution; industrial_production_index only if the text also stresses tanker/port/energy/logistics activity.
- &quot;Введение уголовного наказания за отрицание геноцида...&quot; -&gt; crime_count and mortality_rate.
- &quot;Более 100 человек обратились в больницу с кишечной инфекцией&quot; -&gt; life_expectancy and hospitals.
- &quot;Два человека ранены при атаке БПЛА&quot; -&gt; crime_count and hospitals; life_expectancy may be context, not direct.

Main direct factors:

1. crime_count
- Mark for crime, law enforcement, criminal/court cases, punishment, detention/arrest, fraud, corruption, attack, shelling, drone strike, sabotage, terrorism, military/security incident, strategic-facility disruption, prison/convicts, genocide/crimes memory, legal definition of crimes.
- If there is a market/infrastructure failure with security/operational-risk framing, mark crime_count only when no better factor explains the signal.

2. enterprises_count
- Mark for companies, banks, business actors, plants as economic/legal actors, exports/contracts, bankruptcy, liquidation, profit/loss, lawsuits against companies, investment quotas, corporate plans, transport operators, business associations, named commercial organizations.
- State industrial/economic policy affecting firms also counts.
- Do not mark for every government news unless a business/enterprise/market actor is visible.

3. industrial_production_index
- Mark for plant/factory, extraction, oil/gas/coal/uranium/LNG, electricity and energy infrastructure, production volumes, mining, refinery, aircraft/vehicles/equipment, industrial exports/imports, ports, tankers, railways, airports, commodity logistics, industrial accident, production launch/stop.
- If corporate news is about producing/extracting/exporting industrial goods or energy, usually mark both enterprises_count and industrial_production_index.

4. consumer_price_index
- Mark for explicit prices, inflation, tariffs, fuel/food/utility prices, central-bank rate, duties, oil/gas/coal prices, exchange rate, food markets.
- Also mark for direct supply/production/trade/transport/weather shocks that can plausibly affect prices of consumer goods or services.
- Do not mark for generic politics without economic, trade, commodity, tariff, rate, production, logistics, or commerce channel.

5. mortality_rate
- Mark for killed/dead/fatalities/death toll.
- Also mark for genocide/death memory, war casualties/losses, severe fire/explosion/attack, and death-risk hazards where the death topic is central.
- Do not mark every injury as mortality_rate unless deaths, fatal risk, genocide, war losses, or severe disaster are central.

Secondary direct factors:
- life_expectancy: disease outbreak, mass poisoning/infection, injuries, health risk, dangerous event affecting health.
- hospitals: hospital, ambulance, hospitalization, emergency medical care, people brought/transferred to doctors.
- qualified_doctors: doctors, physicians, medical staff, ambulance doctor.
- air_pollution: smoke, emissions, oil spill/fire, fuel contamination, environmental contamination affecting air/coast.
- wastewater_discharge: sewerage, wastewater, polluted water discharge, water utility accident.
- housing_area_per_capita: damaged homes, housing repair/resettlement, housing utilities, property/living-space rules.
- unemployment_rate: layoffs, job bans, labor-market restrictions.
- per_capita_income: wages, pensions, payments, taxes, compensation, household money.

Strict migration rule:
- international_inflow/outflow only for physical people entering/leaving a country: migrants, tourists, visa flows, prisoners/refugees crossing border.
- internal_arrivals/departures only for physical people moving/evacuating inside a country.
- Never use migration for exports/imports, foreign companies, diplomacy, ships, aircraft restrictions, money flows, markets, or investments.

Relevance:
- 0.90-0.95: main direct/gold-style signal.
- 0.75-0.89: secondary direct signal.
- 0.60-0.74: visible but weaker direct signal.
- Do not use relevance below 0.60.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;security/legal signal -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no visible direct analytical signal exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_core_filtered_v3</code> · v4 direct search · 4210 chars</summary>

<pre># direct_core_filtered_v3

Task: annotate Russian news titles/texts with DIRECT social-signal factors.

Think briefly, then output JSON only. No long reasoning outside JSON.

Goal: maximize direct factor F1 for this dataset. Prefer the core direct factors; avoid rare factors unless the news literally states them.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Primary direct factors to check first:

1. crime_count
- Mark for crime, law enforcement, court/criminal case, punishment, detention/arrest, fraud/corruption, attack, shelling, drone strike, sabotage, terrorism, military/security incident, conflict around strategic facilities, prison/convicts, genocide/crimes memory.
- If people are injured/killed by attack/shelling/crime, also mark crime_count.

2. enterprises_count
- Mark for company, bank, enterprise, corporation, plant as business actor, joint venture, export contract, bankruptcy, liquidation, profit/loss, corporate lawsuit, production abroad, investment quota, business plans, named business actors.

3. industrial_production_index
- Mark for factory/plant, extraction, oil/gas/coal/LNG, electricity/power, production volumes, mining, refinery, aircraft/vehicles/equipment supply, energy infrastructure, industrial accident, port/tanker/commodity logistics.

4. consumer_price_index
- Mark for explicit price/inflation/tariff/rate/duty/fuel/food/utility cost.
- Also mark for oil/gas/coal prices, central-bank rate, import duties, sanctions/trade restrictions, weather/transport/industrial supply disruption when the text plausibly affects consumer price pressure.

5. mortality_rate
- Mark for killed/dead/fatalities, death toll, genocide/death memory, war/conflict casualties, severe fire/explosion/attack with possible deaths.

Secondary direct factors:
- life_expectancy: injured, disease, health threat, accident victims, dangerous event affecting health.
- hospitals: ambulance, hospitalization, emergency medical care, hospitals, inpatient care.
- air_pollution: smoke, emissions, oil spill/fire, environmental contamination affecting air.
- wastewater_discharge: sewerage, wastewater, water supply accident, polluted water discharge, water utility failure.

Suppress noisy rare factors:
- Do NOT mark per_capita_income, real_income_index, maternity_capital, childcare_allowance, housing_area_per_capita, outpatient_clinics, qualified_doctors, unemployment_rate, population/male/female/marriage/divorce/birth factors unless the news literally states that exact social topic.
- Do NOT mark migration factors unless physical people enter/leave/move/evacuate. Never use migration for trade, money flows, exports, foreign companies, sanctions, diplomacy, markets, investments, ships, or aircraft restrictions.

Relevance:
- 0.90-0.95 for main direct core signal.
- 0.75-0.89 for secondary direct signal.
- 0.60-0.74 only when the signal is present but weaker.
- Avoid returning more than 3 factors unless the news clearly combines business/industry/security/price or casualties.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;security/crime signal -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no direct analytical signal exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_core_gold_calibrated_v4</code> · v4 direct search · 5097 chars</summary>

<pre># direct_core_gold_calibrated_v4

Task: annotate Russian news titles/texts with DIRECT social-signal factors for this gold dataset.

Think briefly, then output JSON only. No long reasoning outside JSON.

Important calibration: in this dataset, &quot;direct&quot; means a direct analytical signal for the factor, not only literal statistical measurement. Keep the strong core behavior from direct_core_recall_v2, but also catch broad gold-style direct signals when the text gives a clear channel.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Core direct factors:

1. crime_count
- Crime/law/security/violence: criminal case, court punishment, detention/arrest, fraud/corruption, attack, shelling, drone strike, sabotage, military/security incident, conflict around strategic facilities, prison/convicts, genocide/crimes memory, legal definition of crimes.
- Also mark for blocked rotations/threats around nuclear/strategic facilities or severe geopolitical/security escalation.

2. enterprises_count
- Company, bank, enterprise, corporation, plant as business actor, joint venture, export contract, bankruptcy, liquidation, profit/loss, corporate lawsuit, production abroad, investment quota, business plans, named business actors.
- Also mark for state industrial/economic policy affecting firms, investment in sectors, foreign orders for aircraft/equipment, business/entrepreneurship/commerce as a social-economic signal.

3. industrial_production_index
- Factory/plant, extraction, oil/gas/coal/LNG, electricity/power, production volumes, mining, refinery, aircraft/vehicles/equipment supply, energy infrastructure, industrial accident, port/tanker/commodity logistics.
- Also mark for industrial supply constraints, weather/transport disruptions affecting production, and major commodity/energy market news.

4. consumer_price_index
- Explicit price/inflation/tariff/rate/duty/fuel/food/utility cost.
- Also mark gold-style price-pressure signals: oil/gas/coal prices, central-bank rate, import duties, sanctions/trade restrictions, stock/exchange market with inflation/rate/oil context, weather that can affect food/energy, transport disruption, industrial supply constraints, consumer/commerce/entrepreneurship context.

5. mortality_rate
- Killed/dead/fatalities, death toll.
- Also mark for genocide/death memory, war/conflict casualties or losses, severe fire/explosion/attack, social/legal memory of mass death, or conflict described as a source of fatal risk even if current deaths are not counted.

Secondary direct factors:
- life_expectancy: injured, illness, health risk, disease, accident victims, dangerous event affecting health.
- hospitals: ambulance, hospitalization, emergency medical care, hospitals, inpatient care.
- qualified_doctors: doctors, physicians, medical staff, ambulance doctor.
- air_pollution: smoke, emissions, oil spill/fire, environmental contamination affecting air.
- wastewater_discharge: sewerage, wastewater, water supply accident, polluted water discharge, water utility failure.
- housing_area_per_capita: damaged homes, housing repair/resettlement, ЖКХ/housing utility living conditions.

Rare factors:
- Mark per_capita_income / real_income_index only for explicit wages, pensions, household money, debt, purchasing power, income burden.
- Mark benefits/family/demography factors only for literal benefits, births, marriages, divorces, population composition.
- Migration factors only for physical people entering/leaving/moving/evacuating. Never use migration for trade, money flows, exports, foreign companies, sanctions, diplomacy, markets, investments, ships, or aircraft restrictions.

Relevance:
- 0.90-0.95 for main direct core/gold-style signal.
- 0.75-0.89 for secondary direct signal.
- 0.60-0.74 for plausible but weaker gold-style direct signal.
- Typical relevant news has 1-3 factors; complex business/industry/security/price news can have 3-5.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;security/legal signal -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no analytical signal exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_core_gold_fewshot_v5</code> · v4 direct search · 5648 chars</summary>

<pre># direct_core_gold_fewshot_v5

Task: annotate Russian news titles/texts with DIRECT social-signal factors for this gold dataset.

Think briefly, then output JSON only. No long reasoning outside JSON.

Use the same core factor logic as the best direct prompt, but calibrate to the dataset examples below. In this dataset, `direct` can mean a direct analytical social signal, not only a literal statistical measurement.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Gold-style examples to imitate:
- &quot;Минэнерго ждет снижения добычи угля...&quot; -&gt; industrial_production_index, enterprises_count, consumer_price_index.
- &quot;Глава Минпромторга сообщил о запросах из-за рубежа на поставки Ил-76&quot; -&gt; industrial_production_index, enterprises_count, consumer_price_index.
- &quot;Старт вечерней сессии на срочном рынке Мосбиржи задерживается&quot; -&gt; crime_count as a security/operational-risk signal.
- &quot;Введение уголовного наказания за отрицание геноцида...&quot; -&gt; crime_count, mortality_rate.
- &quot;Пожар в бизнес-центре...&quot; -&gt; mortality_rate as a severe hazard/death-risk signal; also life_expectancy if victims/health threat are mentioned.
- &quot;Президент РФ ... преступления нацистского режима...&quot; -&gt; crime_count; mortality_rate if genocide/death memory is central.
- &quot;В Литву прибыли освобожденные заключенные&quot; -&gt; crime_count; migration only if physical movement across country border is explicit.

Core direct factors:

1. crime_count
- Mark for crime, law enforcement, court/criminal case, punishment, detention/arrest, fraud/corruption, attack, shelling, drone strike, sabotage, terrorism, military/security incident, strategic-facility disruption, prison/convicts, genocide/crimes memory, legal definition of crimes, operational/security failure.
- If people are injured/killed by attack/shelling/crime, mark crime_count in addition to mortality_rate/life_expectancy.

2. enterprises_count
- Mark for company, bank, business, enterprise, corporation, plant as legal/economic actor, joint venture, export contract, bankruptcy, liquidation, profit/loss, corporate lawsuit, production abroad, investment quota, business plans, named business actors.
- State industrial/economic policy affecting firms also counts.

3. industrial_production_index
- Mark for factory/plant, extraction, oil/gas/coal/LNG, electricity/power generation, production volumes, mining, refinery, aircraft/vehicles/equipment supply, energy infrastructure, industrial accident, port/tanker/commodity logistics.
- If corporate news is about producing/extracting/exporting industrial goods or energy, usually mark both enterprises_count and industrial_production_index.

4. consumer_price_index
- Mark for explicit price, inflation, tariff, fuel/food/utility price, central-bank rate, import duty, oil/gas/coal price.
- Also mark for direct supply/production/trade/transport/weather shocks that plausibly affect consumer price pressure.
- Do not mark for generic politics unless there is an economic, trade, commodity, tariff, rate, production, or commerce channel in the text.

5. mortality_rate
- Mark for killed/dead/fatalities/death toll.
- Also mark for genocide/death memory, war casualties/losses, severe fire/explosion/attack, and direct death-risk hazards even if current deaths are not counted.

Secondary factors:
- life_expectancy: injured, disease, health risk, accident victims, dangerous event affecting health.
- hospitals: ambulance, hospitalization, emergency medical care, hospitals, inpatient care.
- qualified_doctors: doctors, physicians, medical staff, ambulance doctor.
- air_pollution: smoke, emissions, oil spill/fire, environmental contamination affecting air.
- wastewater_discharge: sewerage, wastewater, water supply accident, polluted water discharge, water utility failure.
- housing_area_per_capita: damaged homes, housing repair/resettlement, ЖКХ/housing utility living conditions.

Strict rules:
- Migration factors only for physical people entering/leaving/moving/evacuating. Never use migration for trade, money flows, exports, foreign companies, sanctions, diplomacy, markets, investments, ships, or aircraft restrictions.
- Rare demographic/family/income factors require literal mention of that social topic.

Relevance:
- 0.90-0.95 for main direct/gold-style signal.
- 0.75-0.89 for secondary direct signal.
- 0.60-0.74 only when the signal is present but weaker.
- Typical relevant news has 1-3 factors; complex business/industry/security/price news can have 3-5.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;security/legal signal -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no analytical signal exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_core_recall_v2</code> · v4 direct search · 4456 chars</summary>

<pre># direct_core_recall_v2

Task: annotate Russian news titles/texts with DIRECT social-signal factors.

Think briefly, then output JSON only. No long reasoning outside JSON.

Goal: high recall on direct factors without adding weak geopolitical implications. The title is valid evidence.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Always check these high-frequency direct factors:

1. enterprises_count
- Mark for company, bank, business, enterprise, corporation, joint venture, export contract, bankruptcy, liquidation, profit/loss, lawsuit against a company, production abroad, investment quota, corporate plan/project.
- If the news is about a named business actor or business operation, usually mark enterprises_count.

2. crime_count
- Mark for any crime/security/violence/law-enforcement signal: attack, shelling, drone strike, sabotage, terrorism, explosion caused by attack, shooting, fraud, corruption, criminal case, detention, arrest, court sentence, illegal action, investigation, police/FSB/prosecutor.
- If people are injured/killed by attack/shelling/crime, mark crime_count in addition to mortality_rate/life_expectancy.

3. industrial_production_index
- Mark for plant/factory, extraction, LNG/gas/oil/coal, electricity/power generation, industrial output, production volumes, industrial infrastructure, production launch/stop, mining, refinery, energy facility, manufacturing.
- If a corporate news item is about producing/extracting/exporting industrial goods or energy, mark both enterprises_count and industrial_production_index.

4. consumer_price_index
- Mark for price, inflation, tariff, fuel price, food price, utility cost, electricity/gas/heating/water tariff, cost of consumer goods/services.
- Do not infer prices from sanctions or trade unless the text explicitly mentions prices/tariffs/costs/inflation.

Other direct mappings:
- killed, dead, fatality, death toll -&gt; mortality_rate
- injured, disease, life/health threat -&gt; life_expectancy
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals
- doctor, physician, medical worker -&gt; qualified_doctors
- housing damage, home repair/resettlement, living space, housing utilities -&gt; housing_area_per_capita
- air emissions, smoke, air pollution -&gt; air_pollution
- sewage, wastewater, polluted water discharge -&gt; wastewater_discharge
- birth, newborn, fertility -&gt; birth_rate_per_1000
- infant death -&gt; infant_mortality and mortality_rate
- unemployment, layoffs, job loss -&gt; unemployment_rate
- wage, salary, pension, household payment -&gt; per_capita_income
- purchasing power, income eroded by inflation, debt burden -&gt; real_income_index

Strict migration rule:
- Use international_inflow/outflow only for physical people migrating or entering/leaving a country.
- Use internal_arrivals/departures only for physical people moving/evacuating inside the country.
- Do NOT use migration factors for exports/imports, foreign companies, sanctions, diplomacy, ships, aircraft restrictions, money flows, markets, investments, or international cooperation.

Relevance:
- 0.85-1.00 main direct signal.
- 0.60-0.84 secondary direct signal.
- Do not use relevance below 0.60.
- Typical relevant news has 1-3 direct factors; complex attacks/industrial-business news can have 3-5.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;attack/criminal event -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no direct factor exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_core_targeted_v7</code> · v4 direct search · 7044 chars</summary>

<pre># direct_core_targeted_v7

Task: annotate Russian news titles/texts with DIRECT social-signal factors.

Think briefly, then output JSON only. No prose outside JSON.

Goal: maximize direct micro-F1 on the v4 gold dataset. Use direct analytical links, but avoid broad weak/context links.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Calibrated direct rules:

1. crime_count
- Mark for crime, court/criminal case, criminal punishment, detention/arrest, fraud, corruption, law enforcement, prosecutor/FSB/police, prison/convicts, attack, shelling, drone strike, sabotage, terrorism, military/security incident, strategic-facility disruption.
- Also mark for genocide/crimes memory or legal definition of crimes.
- Do not mark ordinary economic, stock-market, health, ecology, or transport news unless the text has security/law-enforcement/attack/operational-risk framing.

2. enterprises_count
- Mark for company, bank, business, enterprise, corporation, plant as economic actor, transport operator, export/import contract, bankruptcy, liquidation, profit/loss, corporate lawsuit, investment quota, corporate production/business plan, named commercial organization.
- State policy counts only when it directly affects firms/markets/enterprise activity.
- Do not mark a factor merely because some organization is mentioned; there must be a business/market/enterprise channel.

3. industrial_production_index
- Mark for plant/factory, extraction, oil/gas/coal/uranium/LNG, electricity/power generation, production volume, mining, refinery, industrial goods, aircraft/vehicles/equipment, energy infrastructure, industrial exports/imports, ports, tankers, railways, airports, logistics shutdowns, production launch/stop.
- Mark for commodity/resource policy if it affects extraction, production, logistics, or industrial supply.
- If a corporate news item is about producing/extracting/exporting industrial goods or energy, usually mark both enterprises_count and industrial_production_index.

4. consumer_price_index
- Mark for explicit prices, inflation, tariffs, fuel/food/utility prices, central-bank rate, import duty, oil/gas/coal price, exchange rate, food markets.
- Also mark direct supply/logistics/production shocks if they clearly affect consumer prices: food, fuel, energy, tariffs, transport costs, commodity prices.
- Do not mark symbolic/cultural/political news as CPI unless the text has a visible price, rate, tariff, commodity, supply, production, logistics, trade, or consumer-cost channel.

5. mortality_rate
- Mark for killed/dead/fatalities/death toll.
- Also mark genocide/death memory, war casualties/losses, severe fire/explosion/attack, and death-risk disasters when the death topic or fatal hazard is central.
- Do not mark routine market/economic news as mortality_rate unless it explicitly references deaths, war losses, genocide, fatality, or severe disaster.

Medical/ecology/housing:
- life_expectancy: mass illness, poisoning, infection, injuries, health threat, dangerous event affecting health.
- hospitals: hospital, ambulance, hospitalization, emergency medical care, people brought/transferred to doctors.
- qualified_doctors: doctors, physicians, medical staff, ambulance doctor.
- air_pollution: smoke, emissions, oil spill/fire, fuel contamination, dirty air, environmental contamination of coast/air.
- wastewater_discharge: sewage, wastewater, polluted water discharge, water utility accident.
- housing_area_per_capita: damaged homes, housing repair/resettlement, living space, housing utilities, property/housing transaction rules.

Income/labor/demography:
- unemployment_rate: layoffs, job loss, labor bans/restrictions, unemployment.
- per_capita_income: wages, pensions, payments, taxes, compensation, household money.
- real_income_index: purchasing power or income explicitly linked to prices/inflation/debt burden.
- family/demographic factors require literal mention of births, marriages, divorces, children benefits, maternity capital, abortions, infant deaths, population size.

Strict migration rule:
- international_inflow/outflow only for physical people entering/leaving a country: migrants, tourists/visas, refugees, released prisoners crossing border.
- internal_arrivals/departures only for physical people moving/evacuating inside the country.
- Never use migration for exports/imports, foreign companies, sanctions, diplomacy, ships, aircraft restrictions, money flows, markets, investments, or generic international cooperation.

Positive examples:
- &quot;Старт вечерней сессии на срочном рынке Мосбиржи задерживается&quot; -&gt; crime_count.
- &quot;Аэропорты Волгограда и Краснодара приостановили работу&quot; -&gt; industrial_production_index.
- &quot;Запросы из-за рубежа на поставки Ил-76&quot; -&gt; industrial_production_index, enterprises_count, consumer_price_index.
- &quot;Мировые цены на продовольствие выросли&quot; -&gt; consumer_price_index, industrial_production_index.
- &quot;Более 100 человек обратились в больницу с кишечной инфекцией&quot; -&gt; life_expectancy, hospitals.
- &quot;Уголовное наказание за отрицание геноцида&quot; -&gt; crime_count, mortality_rate.

Negative examples:
- Do not use international_inflow/outflow for airport shutdown, queues, exports, foreign firms, or sanctions unless people crossing a country border are the topic.
- Do not use consumer_price_index for monuments/culture unless commerce/price channel is explicit.
- Do not use mortality_rate for ordinary stock-market news unless death/war/genocide/fatal risk is explicit.
- Do not use hospitals for &quot;people saved&quot; unless doctors/hospital/ambulance/medical transfer is explicit.

Relevance:
- 0.90-0.95 for main direct signal.
- 0.75-0.89 for secondary direct signal.
- 0.60-0.74 for visible but weaker direct signal.
- Do not use relevance below 0.60.
- Typical relevant news has 1-3 direct factors; complex industrial/security/business/price news can have 3-5.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.7,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;security/legal signal -&gt; crime_count&quot;
        }
      ]
    }
  ]
}

If no direct factor exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>direct_fewshot</code> · v4 direct search · 4314 chars</summary>

<pre># direct_fewshot

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.

Examples:
- &quot;погибли два человека при пожаре&quot; -&gt; mortality_rate, crime_count if fire/attack/crime context, life_expectancy if injuries/threat.
- &quot;тарифы ЖКХ выросли&quot; -&gt; consumer_price_index.
- &quot;завод остановил производство&quot; -&gt; industrial_production_index, enterprises_count.
- &quot;банк лишился лицензии&quot; -&gt; enterprises_count.
- &quot;переговоры РФ-США&quot; -&gt; annotations=[] unless a direct factor event is stated.

Hard negative rules:
- Do not mark international_inflow/outflow for diplomacy, sanctions, exports/imports, negotiations, documents, ships, logistics, or trade unless people physically move.
- Do not mark consumer_price_index for stock prices, IPO, company revenue, commodity quotes, or exchange news without a channel to consumer costs/tariffs/inflation.
- Do not mark industrial_production_index for a generic company, bank, investment, or financial transaction without production/extraction/infrastructure/output.
- Do not mark hospitals for water/electricity/utility accidents unless medical care, ambulance, hospitalization, or medical institutions are mentioned.
- Do not mark air_pollution for weather, temperature, rain, cold, or forecasts without pollution/smoke/emissions.
- Do not mark mortality_rate when the text explicitly says there are no casualties.
- Do not mark child/family factors from the word &quot;family&quot; alone.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_minimal</code> · v4 direct search · 3045 chars</summary>

<pre># direct_minimal

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.

Return the smallest complete direct set. No weak/context factors.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_negative</code> · v4 direct search · 3989 chars</summary>

<pre># direct_negative

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.


Hard negative rules:
- Do not mark international_inflow/outflow for diplomacy, sanctions, exports/imports, negotiations, documents, ships, logistics, or trade unless people physically move.
- Do not mark consumer_price_index for stock prices, IPO, company revenue, commodity quotes, or exchange news without a channel to consumer costs/tariffs/inflation.
- Do not mark industrial_production_index for a generic company, bank, investment, or financial transaction without production/extraction/infrastructure/output.
- Do not mark hospitals for water/electricity/utility accidents unless medical care, ambulance, hospitalization, or medical institutions are mentioned.
- Do not mark air_pollution for weather, temperature, rain, cold, or forecasts without pollution/smoke/emissions.
- Do not mark mortality_rate when the text explicitly says there are no casualties.
- Do not mark child/family factors from the word &quot;family&quot; alone.

Final self-check: remove every factor whose evidence is only broad context.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_precision</code> · v4 direct search · 3988 chars</summary>

<pre># direct_precision

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.


Hard negative rules:
- Do not mark international_inflow/outflow for diplomacy, sanctions, exports/imports, negotiations, documents, ships, logistics, or trade unless people physically move.
- Do not mark consumer_price_index for stock prices, IPO, company revenue, commodity quotes, or exchange news without a channel to consumer costs/tariffs/inflation.
- Do not mark industrial_production_index for a generic company, bank, investment, or financial transaction without production/extraction/infrastructure/output.
- Do not mark hospitals for water/electricity/utility accidents unless medical care, ambulance, hospitalization, or medical institutions are mentioned.
- Do not mark air_pollution for weather, temperature, rain, cold, or forecasts without pollution/smoke/emissions.
- Do not mark mortality_rate when the text explicitly says there are no casualties.
- Do not mark child/family factors from the word &quot;family&quot; alone.

Only return a factor if you can quote evidence from the text in evidence.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_recall</code> · v4 direct search · 4016 chars</summary>

<pre># direct_recall

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.


Hard negative rules:
- Do not mark international_inflow/outflow for diplomacy, sanctions, exports/imports, negotiations, documents, ships, logistics, or trade unless people physically move.
- Do not mark consumer_price_index for stock prices, IPO, company revenue, commodity quotes, or exchange news without a channel to consumer costs/tariffs/inflation.
- Do not mark industrial_production_index for a generic company, bank, investment, or financial transaction without production/extraction/infrastructure/output.
- Do not mark hospitals for water/electricity/utility accidents unless medical care, ambulance, hospitalization, or medical institutions are mentioned.
- Do not mark air_pollution for weather, temperature, rain, cold, or forecasts without pollution/smoke/emissions.
- Do not mark mortality_rate when the text explicitly says there are no casualties.
- Do not mark child/family factors from the word &quot;family&quot; alone.

Be recall-oriented inside direct-only rules: if a direct event clearly touches two factors, return both.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_strict</code> · v4 direct search · 3964 chars</summary>

<pre># direct_strict

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.


Hard negative rules:
- Do not mark international_inflow/outflow for diplomacy, sanctions, exports/imports, negotiations, documents, ships, logistics, or trade unless people physically move.
- Do not mark consumer_price_index for stock prices, IPO, company revenue, commodity quotes, or exchange news without a channel to consumer costs/tariffs/inflation.
- Do not mark industrial_production_index for a generic company, bank, investment, or financial transaction without production/extraction/infrastructure/output.
- Do not mark hospitals for water/electricity/utility accidents unless medical care, ambulance, hospitalization, or medical institutions are mentioned.
- Do not mark air_pollution for weather, temperature, rain, cold, or forecasts without pollution/smoke/emissions.
- Do not mark mortality_rate when the text explicitly says there are no casualties.
- Do not mark child/family factors from the word &quot;family&quot; alone.

Be conservative: precision matters more than recall.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>direct_taxonomy</code> · v4 direct search · 3153 chars</summary>

<pre># direct_taxonomy

You annotate Russian news for a social-signal factor dataset.

Allowed factor_id values:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

Direct-only target:
- Return a factor only when the news text directly describes the event represented by that factor.
- Do not return weak macro implications, generic political context, or long causal chains.
- Typical direct positive count is 0-2 factors. Complex accidents/attacks can have 3-5.
- Use relevance &gt;= 0.80 for a direct main signal and 0.55-0.79 for a direct secondary signal.
- Do not use relevance below 0.55 in direct-only prompts.

Direct examples:
- death, killed, fatalities -&gt; mortality_rate.
- injured, threat to life/health, disease outbreak -&gt; life_expectancy.
- hospital, ambulance, hospitalization, inpatient care -&gt; hospitals.
- doctors, medical staff, shortage of doctors -&gt; qualified_doctors.
- attack, crime, fraud, corruption, criminal case, violence -&gt; crime_count.
- consumer prices, tariffs, inflation, fuel/food/utility prices -&gt; consumer_price_index.
- wages, pensions, household payments, personal income -&gt; per_capita_income.
- purchasing power, inflation pressure on income, debts/credit burden -&gt; real_income_index.
- production, extraction, plant, industrial infrastructure, energy output -&gt; industrial_production_index.
- company, bank, business opening/closure, bankruptcy, corporate activity -&gt; enterprises_count.
- damaged housing, resettlement, housing conditions, utilities in homes -&gt; housing_area_per_capita.
- water supply, sewerage, wastewater, water pollution -&gt; wastewater_discharge.
- smoke, emissions, air pollution, oil spill/fire with air/ecological damage -&gt; air_pollution.
- actual migration or evacuation of people -&gt; migration factors. Diplomacy/trade is not migration.

Treat factor names as event categories, not statistical formulas. Example: mortality_rate means deaths/fatalities; hospitals means hospitalization/ambulance/inpatient care.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;crime_count&quot;,
          &quot;relevance&quot;: 0.8,
          &quot;sentiment&quot;: &quot;negative&quot;,
          &quot;pressure&quot;: 0.6,
          &quot;confidence&quot;: 0.8,
          &quot;evidence&quot;: &quot;short quote from news&quot;,
          &quot;reason&quot;: &quot;attack -&gt; crime_count&quot;
        }
      ]
    }
  ]
}
Use only factor_id from the list. If no factor is relevant, annotations=[].
</pre>

</details>

<details><summary><code>gold_mimic_broad_v1</code> · v4 direct search · 4536 chars</summary>

<pre># gold_mimic_broad_v1

Task: annotate Russian news for the current gold dataset logic.

Think briefly, then output JSON only. No long reasoning outside JSON.

Important: in this dataset, `direct` often means &quot;the news is a direct analytical social signal for this factor&quot;, not only a literal statistical measurement. Use broad but text-grounded links. Prefer recall over strictness. Typical relevant news has 2-4 factors.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000

High-frequency gold logic:

- enterprises_count: company, bank, business, enterprise, corporation, plant/project, joint venture, export contract, bankruptcy, liquidation, profit/loss, corporate lawsuit, production abroad, investment quota, business plans. Also state industrial/economic policy affecting firms.

- industrial_production_index: plant/factory, extraction, oil/gas/coal/LNG, electricity/power, industrial output, production volumes, mining, refinery, aircraft/vehicles/equipment supply, energy infrastructure, industrial accident, port/tanker/commodity logistics.

- consumer_price_index: explicit prices/inflation/tariffs/rates/duties/fuel/food/utilities, plus social/economic signals likely to affect consumer price pressure: oil/coal/gas prices, exchange/stock market, central-bank rate, import duties, sanctions/trade restrictions, weather affecting food/energy, transport disruption, industrial supply constraints, entrepreneurship/commerce signals.

- crime_count: crime, law enforcement, court/criminal case, detention, punishment, corruption/fraud, attack, shelling, drone strike, sabotage, military/security incident, conflict escalation, border/security tension, prison/convicts, genocide/crimes memory, blocked rotations or threats around strategic facilities.

- mortality_rate: killed/dead/fatalities, genocide/death memory, war/conflict casualties, severe fire/explosion/attack, social signals of death/loss/threat even when deaths are historical or implied.

- life_expectancy: injuries, illness, health risk, disease, accident victims, dangerous environmental/war/industrial event.

- hospitals: ambulance, hospitalization, emergency medical care, hospitals, inpatient care.

- qualified_doctors: doctors, physicians, medical workers, ambulance doctor, shortage/staffing of medical personnel.

- housing_area_per_capita: damaged homes, housing repair/resettlement, utilities in homes, ЖКХ debt/quality, living conditions.

- air_pollution: smoke, emissions, oil spill/fire, environmental contamination affecting air.

- wastewater_discharge: sewerage, wastewater, water supply accident, polluted water discharge, water utility failure.

- unemployment_rate: layoffs, job cuts, labor market trouble, business closures affecting jobs.

- per_capita_income: wages, salaries, pensions, household payments, benefits, income, dividends/household money.

- real_income_index: purchasing power, inflation burden, debt/credit burden, income erosion.

- migration factors: only physical people entering/leaving/moving/evacuating; do not use for trade, money flows, exports, foreign companies.

Relevance:
- Use 0.86-0.95 for a gold-like analytical signal.
- Use 0.70-0.85 for secondary but plausible signal.
- Avoid relevance below 0.60.
- If a news item has a business/industry/security/price channel, usually return all applicable core factors.

Return only valid JSON:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: 1,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;consumer_price_index&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;neutral&quot;,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.85,
          &quot;evidence&quot;: &quot;short quote&quot;,
          &quot;reason&quot;: &quot;gold-like price pressure signal&quot;
        }
      ]
    }
  ]
}

If no analytical signal exists, use &quot;annotations&quot;: [].
</pre>

</details>

<details><summary><code>high_recall_minimal_v1</code> · v4 direct search · 3167 chars</summary>

<pre>Разметь новости по социально-экономическим факторам.

Главная цель: HIGH RECALL. Лучше вернуть лишнюю потенциальную связь с низкой оценкой, чем пропустить фактор.

Шкала:
- 0.90: прямой главный сигнал.
- 0.60: явная связь.
- 0.30: слабая, но возможная аналитическая связь.

Верни JSON:
{&quot;items&quot;:[{&quot;dataset_row_id&quot;:1,&quot;exclude&quot;:false,&quot;exclude_reason&quot;:&quot;&quot;,&quot;no_factor_reason&quot;:&quot;&quot;,&quot;annotations&quot;:[{&quot;factor_id&quot;:&quot;crime_count&quot;,&quot;relevance&quot;:0.6,&quot;sentiment&quot;:&quot;negative&quot;,&quot;pressure&quot;:0.5,&quot;confidence&quot;:0.8,&quot;evidence&quot;:&quot;короткая фраза&quot;,&quot;reason&quot;:&quot;почему фактор связан&quot;}]}]}

Факторы:
- population_size: население, демография, численность.
- divorce_rate: разводы.
- marriage_rate: браки.
- international_inflow: въезд людей из-за рубежа, мигранты, туристы, беженцы.
- international_outflow: выезд людей за рубеж, эмиграция, эвакуация.
- internal_arrivals: прибытие людей внутри страны.
- internal_departures: отъезд людей внутри страны.
- infant_mortality: смерть младенцев, риски для младенцев.
- life_expectancy: здоровье, травмы, ранения, угрозы жизни, качество жизни.
- male_population: мужчины как группа населения.
- female_population: женщины как группа населения.
- per_capita_income: доходы, выплаты, зарплаты, деньги населения.
- real_income_index: реальные доходы, покупательная способность.
- unemployment_rate: занятость, безработица, рабочие места.
- living_wage: прожиточный минимум, минимальное обеспечение.
- child_living_wage: прожиточный минимум детей.
- hospitals: больницы, госпитализация, скорая, медучреждения.
- outpatient_clinics: поликлиники, амбулаторная помощь.
- abortions: аборты.
- qualified_doctors: врачи, медики, медперсонал, дефицит кадров.
- preschool_coverage: детские сады, дошкольные места.
- children_benefits: выплаты детям, пособия семьям.
- maternity_capital: материнский капитал.
- childcare_allowance: уход за ребенком, выплаты по уходу.
- large_families_housing: жилье многодетным.
- housing_area_per_capita: жилье, дома, расселение, ЖКХ, коммунальные аварии.
- consumer_price_index: цены, тарифы, инфляция, стоимость товаров/услуг.
- primary_housing_price_index: цены новостроек.
- secondary_housing_price_index: цены вторичного жилья.
- industrial_production_index: производство, добыча, заводы, энергия, транспортная/портовая логистика.
- enterprises_count: компании, бизнес, банки, банкротства, открытие/закрытие, прибыль, инвестиции.
- crime_count: преступления, атаки, обстрелы, терроризм, мошенничество, коррупция, безопасность.
- air_pollution: выбросы, дым, пожар, разлив топлива, загрязнение воздуха/среды.
- wastewater_discharge: вода, стоки, водоснабжение, загрязнение воды.
- mortality_rate: погибшие, умершие, смертельный исход.
- birth_rate_per_1000: рождаемость, роды, поддержка рождения детей.

Минимальные ограничения:
- Миграционные факторы ставь только про движение людей, не товаров.
- Если есть цена/тариф/инфляция, ставь consumer_price_index.
- Если есть компания, промышленность или госполитика для бизнеса, часто ставь enterprises_count и/или industrial_production_index.
- Если есть вред людям, часто ставь life_expectancy; если есть погибшие, mortality_rate; если есть больница, hospitals.
- Не пиши текст вне JSON.
</pre>

</details>

<details><summary><code>high_recall_minimal_v2</code> · v4 direct search · 3783 chars</summary>

<pre>Разметь новости по социально-экономическим факторам.

Главная цель: HIGH RECALL. Лучше вернуть лишнюю слабую связь с 0.30, чем пропустить фактор.

Шкала только такая:
- 0.90: прямой главный сигнал.
- 0.60: явная связь из текста.
- 0.30: слабая гипотеза; не повышай до 0.60.

Верни только валидный JSON:
{&quot;items&quot;:[{&quot;dataset_row_id&quot;:1,&quot;exclude&quot;:false,&quot;exclude_reason&quot;:&quot;&quot;,&quot;no_factor_reason&quot;:&quot;&quot;,&quot;annotations&quot;:[{&quot;factor_id&quot;:&quot;crime_count&quot;,&quot;relevance&quot;:0.6,&quot;sentiment&quot;:&quot;negative&quot;,&quot;pressure&quot;:0.5,&quot;confidence&quot;:0.8,&quot;evidence&quot;:&quot;короткая фраза&quot;,&quot;reason&quot;:&quot;почему фактор связан&quot;}]}]}

Факторы:
- population_size: население, демография, численность.
- divorce_rate: разводы.
- marriage_rate: браки.
- international_inflow: въезд людей из-за рубежа, мигранты, туристы, беженцы.
- international_outflow: выезд людей за рубеж, эмиграция, эвакуация.
- internal_arrivals: прибытие людей внутри страны.
- internal_departures: отъезд людей внутри страны.
- infant_mortality: смерть младенцев, риски для младенцев.
- life_expectancy: здоровье, травмы, ранения, угрозы жизни, качество жизни.
- male_population: мужчины как группа населения.
- female_population: женщины как группа населения.
- per_capita_income: доходы, выплаты, зарплаты, деньги населения.
- real_income_index: реальные доходы, покупательная способность, потребительские расходы.
- unemployment_rate: занятость, безработица, рабочие места, возврат к работе.
- living_wage: прожиточный минимум, минимальное обеспечение.
- child_living_wage: прожиточный минимум детей.
- hospitals: больницы как система/учреждения; не разовая госпитализация пострадавших.
- outpatient_clinics: поликлиники, амбулаторная помощь.
- abortions: аборты.
- qualified_doctors: врачи, медики, медперсонал, дефицит кадров.
- preschool_coverage: детские сады, дошкольные места.
- children_benefits: выплаты детям, пособия семьям, алименты.
- maternity_capital: материнский капитал.
- childcare_allowance: уход за ребенком, выплаты по уходу.
- large_families_housing: жилье многодетным.
- housing_area_per_capita: жилье, дома, расселение, ЖКХ, коммунальные аварии.
- consumer_price_index: цены, тарифы, инфляция, стоимость товаров/услуг.
- primary_housing_price_index: цены новостроек.
- secondary_housing_price_index: цены вторичного жилья.
- industrial_production_index: производство, добыча, продажи промышленной продукции, заводы, энергия, транспортная/портовая логистика.
- enterprises_count: открытие/закрытие/ликвидация/банкротство компаний, число бизнесов; не любое упоминание компании.
- crime_count: преступления, атаки, обстрелы, терроризм, мошенничество, коррупция, безопасность.
- air_pollution: выбросы, дым, пожар, разлив топлива/нефтепродуктов, загрязнение воздуха/среды.
- wastewater_discharge: вода, стоки, водоснабжение, загрязнение воды.
- mortality_rate: погибшие, умершие, смертельный исход, упоминание что жертв нет.
- birth_rate_per_1000: рождаемость, роды, поддержка рождения детей.

Короткие правила:
- Расходы/покупательная способность -&gt; real_income_index; цены/тарифы/инфляция -&gt; consumer_price_index.
- Продажи автопроизводителя, добыча, квоты, комплектующие, энергопроект -&gt; industrial_production_index 0.60.
- Компания сама по себе не enterprises_count; ставь его только про появление, закрытие, ликвидацию, банкротство или число компаний.
- Атака/обстрел/трагедия -&gt; crime_count; life_expectancy только если есть вред здоровью; mortality_rate если есть погибшие или явно сказано, что жертв/угрозы жизни нет.
- Алименты и детские выплаты -&gt; children_benefits. Шатдаун, рабочие места, возврат ведомств к работе -&gt; unemployment_rate.
- Миграционные факторы ставь только про движение людей, не товаров.
- male_population/female_population ставь только про численность или демографию, не просто болезнь чаще у пола.
- Не пиши текст вне JSON.
</pre>

</details>

<details><summary><code>high_recall_minimal_v3</code> · v4 direct search · 3951 chars</summary>

<pre>Разметь новости по социально-экономическим факторам.

Главная цель: HIGH RECALL. Лучше вернуть лишнюю слабую связь с 0.30, чем пропустить фактор.
Для обычной содержательной новости чаще нужно 2-4 фактора: прямой сигнал + 1-2 контекстных/слабых на 0.30. Один фактор оставляй только если других разумных связей нет.

Шкала только такая:
- 0.90: прямой главный сигнал.
- 0.60: явная связь из текста.
- 0.30: слабая гипотеза; не повышай до 0.60.

Верни только валидный JSON:
{&quot;items&quot;:[{&quot;dataset_row_id&quot;:1,&quot;exclude&quot;:false,&quot;exclude_reason&quot;:&quot;&quot;,&quot;no_factor_reason&quot;:&quot;&quot;,&quot;annotations&quot;:[{&quot;factor_id&quot;:&quot;crime_count&quot;,&quot;relevance&quot;:0.6,&quot;sentiment&quot;:&quot;negative&quot;,&quot;pressure&quot;:0.5,&quot;confidence&quot;:0.8,&quot;evidence&quot;:&quot;короткая фраза&quot;,&quot;reason&quot;:&quot;почему фактор связан&quot;}]}]}

Факторы:
- population_size: население, демография, численность.
- divorce_rate: разводы.
- marriage_rate: браки.
- international_inflow: въезд людей из-за рубежа, мигранты, туристы, беженцы.
- international_outflow: выезд людей за рубеж, эмиграция, эвакуация.
- internal_arrivals: прибытие людей внутри страны.
- internal_departures: отъезд людей внутри страны.
- infant_mortality: смерть младенцев, риски для младенцев.
- life_expectancy: здоровье, травмы, ранения, угрозы жизни, качество жизни.
- male_population: мужчины как группа населения.
- female_population: женщины как группа населения.
- per_capita_income: доходы, выплаты, зарплаты, деньги населения.
- real_income_index: реальные доходы, покупательная способность, потребительские расходы.
- unemployment_rate: занятость, безработица, рабочие места, возврат к работе.
- living_wage: прожиточный минимум, минимальное обеспечение.
- child_living_wage: прожиточный минимум детей.
- hospitals: больницы как система/учреждения; не разовая госпитализация пострадавших.
- outpatient_clinics: поликлиники, амбулаторная помощь.
- abortions: аборты.
- qualified_doctors: врачи, медики, медперсонал, дефицит кадров.
- preschool_coverage: детские сады, дошкольные места.
- children_benefits: выплаты детям, пособия семьям, алименты.
- maternity_capital: материнский капитал.
- childcare_allowance: уход за ребенком, выплаты по уходу.
- large_families_housing: жилье многодетным.
- housing_area_per_capita: жилье, дома, расселение, ЖКХ, коммунальные аварии.
- consumer_price_index: цены, тарифы, инфляция, стоимость товаров/услуг.
- primary_housing_price_index: цены новостроек.
- secondary_housing_price_index: цены вторичного жилья.
- industrial_production_index: производство, добыча, продажи промышленной продукции, заводы, энергия, транспортная/портовая логистика.
- enterprises_count: открытие/закрытие/ликвидация/банкротство компаний, число бизнесов; не любое упоминание компании.
- crime_count: преступления, атаки, обстрелы, терроризм, мошенничество, коррупция, безопасность.
- air_pollution: выбросы, дым, пожар, разлив топлива/нефтепродуктов, загрязнение воздуха/среды.
- wastewater_discharge: вода, стоки, водоснабжение, загрязнение воды.
- mortality_rate: погибшие, умершие, смертельный исход, упоминание что жертв нет.
- birth_rate_per_1000: рождаемость, роды, поддержка рождения детей.

Короткие правила:
- Расходы/покупательная способность -&gt; real_income_index; цены/тарифы/инфляция -&gt; consumer_price_index.
- Продажи автопроизводителя, добыча, квоты, комплектующие, энергопроект -&gt; industrial_production_index 0.60.
- Компания сама по себе не enterprises_count; ставь его только про появление, закрытие, ликвидацию, банкротство или число компаний.
- Атака/обстрел/трагедия -&gt; crime_count; life_expectancy только если есть вред здоровью; mortality_rate если есть погибшие или явно сказано, что жертв/угрозы жизни нет.
- Алименты и детские выплаты -&gt; children_benefits. Шатдаун, рабочие места, возврат ведомств к работе -&gt; unemployment_rate.
- Миграционные факторы ставь только про движение людей, не товаров.
- male_population/female_population ставь только про численность или демографию, не просто болезнь чаще у пола.
- Не пиши текст вне JSON.
</pre>

</details>

<details><summary><code>social_signal_v9_f1_balanced</code> · v4 direct search · 8126 chars</summary>

<pre># social_signal_v9_f1_balanced

Ты размечаешь русскую новость для датасета социальных сигналов РФ.

Задача: вернуть все factor_id из списка 36, для которых в тексте есть видимый причинный или аналитический канал. Это multi-label задача, не выбор одной рубрики.

Главная цель: высокий micro-F1.

Поэтому:
- не пропускай основные co-labels;
- не добавляй слабую или скрытую связь без evidence;
- используй direct analytical signal: фактор может быть прямым аналитическим социальным сигналом, даже если в новости нет буквального статистического показателя Росстата;
- не используй relevance &lt; 0.60;
- обычная релевантная новость: 1-3 фактора;
- сложная бизнес / промышленность / безопасность / цены: 3-5 факторов;
- максимум 6 факторов.

Allowed factor_id:
population_size, divorce_rate, marriage_rate, international_inflow, international_outflow, internal_arrivals, internal_departures, infant_mortality, life_expectancy, male_population, female_population, per_capita_income, real_income_index, unemployment_rate, living_wage, child_living_wage, hospitals, outpatient_clinics, abortions, qualified_doctors, preschool_coverage, children_benefits, maternity_capital, childcare_allowance, large_families_housing, housing_area_per_capita, consumer_price_index, primary_housing_price_index, secondary_housing_price_index, industrial_production_index, enterprises_count, crime_count, air_pollution, wastewater_discharge, mortality_rate, birth_rate_per_1000.

Не используй not_relevant, other, unclear или новые классы.

Scope:
Включай новости про РФ, российские регионы, население, компании, бюджет, промышленность, транспорт, инфраструктуру, медицину, жилье, безопасность, цены, занятость, доходы, экологию, семьи, детей, миграцию.

Зарубежную новость включай только если есть конкретный канал влияния на РФ: российские компании/граждане/активы, санкции против РФ, экспорт/импорт РФ, нефть/газ/СПГ/уголь/уран/продовольствие, валюты/ставки/цены, логистика, конфликт вокруг Украины, энергетический или торговый канал.

Если нет фактора: annotations=[].

Core factor rules:

1. enterprises_count
Ставь для named company, bank, business, commercial operator, plant as business actor, export/import contract, bankruptcy/liquidation, profit/loss, corporate lawsuit, investment, sanctions/regulation affecting firms.
Обычно ставь вместе с industrial_production_index, если компания производит, добывает, транспортирует, импортирует, экспортирует или поставляет industrial goods, energy, vehicles, fuel, food, raw materials.

2. industrial_production_index
Ставь для factory/plant, extraction, oil/gas/coal/уран/LNG, electricity/energy infrastructure, mining/refinery, production volume, industrial goods, aircraft/vehicles/equipment, ports/tankers/rail/airports/logistics shutdown, industrial accident, commodity/resource policy affecting production or supply.

3. consumer_price_index
Ставь для explicit prices, inflation, tariffs, fuel/food/utility prices, key rate, duties, oil/gas/coal/food prices, exchange rate.
Также ставь для direct supply/logistics/production/trade/weather shocks, которые plausibly affect consumer prices.

4. crime_count
Ставь для crime, law enforcement, court/criminal case, punishment, detention/arrest, fraud/corruption, FSB/police/prosecutor, attack/shelling/drone strike/sabotage/terrorism, military/security incident, strategic-facility disruption, prison/convicts, genocide/crimes memory, legal definition of crimes.

5. mortality_rate
Ставь для killed/dead/fatalities/death toll; genocide/death memory; war losses; severe fire/explosion/attack/fatal accident/death-risk disaster, когда death/fatal hazard центральный.

6. life_expectancy
Ставь для injuries, disease outbreak, poisoning, infection, serious health threat, dangerous event affecting health.

7. hospitals
Ставь только если есть hospital, ambulance, hospitalization, emergency medical care, people brought/transferred to doctors.

8. qualified_doctors
Ставь для doctors, physicians, medical staff, ambulance doctor.

9. air_pollution
Ставь для smoke, emissions, oil/fuel contamination, environmental contamination affecting air/coast, major fuel/chemical spill/fire with pollution channel.

10. wastewater_discharge
Ставь для sewerage, wastewater, polluted water discharge, water utility/sewage accident.

11. housing_area_per_capita
Ставь для damaged homes, resettlement, housing repair, housing utilities/living conditions, property/living-space rules.

12. Income/labor:
per_capita_income = wages, pensions, social payments, taxes, compensation, household money.
real_income_index = purchasing power, inflation/debt burden on income.
unemployment_rate = layoffs, job loss, labor restrictions, unemployment.

13. Family/demography factors
Требуют буквального социального топика: births, marriages, divorces, children benefits, maternity capital, childcare, preschool, abortions, infant deaths, population size or sex structure.

Strict negative rules:
- Migration factors only for physical movement of people. Never use international_inflow, international_outflow, internal_arrivals, internal_departures for exports/imports, trade, sanctions, foreign firms, diplomacy, ships, aircraft restrictions, money flows, markets, investments, cargo/logistics without people.
- Do not use hospitals for utility accidents, water supply, rescue, injuries, or &quot;people saved&quot; unless hospital/ambulance/doctors/medical transfer is explicit.
- Do not use mortality_rate for ordinary injury/market/economic news unless deaths, fatal risk, genocide, war losses, or severe disaster are central.
- Do not use industrial_production_index for pure stock movement, finance, IPO, valuation, court, culture, or politics without production/extraction/energy/logistics/supply.
- Do not use consumer_price_index for corporate revenue, asset price, IPO, sanctions, politics, or finance unless consumer prices/rates/tariffs/commodities/supply costs are visible.
- Do not use air_pollution for weather without smoke/emissions/pollution.
- Do not use population_size, male_population, female_population for education, politics, culture, or business unless численность/структура населения is explicit.

Gold calibration examples:
- &quot;Старт вечерней сессии на срочном рынке Мосбиржи задерживается&quot; -&gt; crime_count.
- &quot;Аэропорты Волгограда и Краснодара приостановили работу&quot; -&gt; industrial_production_index.
- &quot;Глава Минпромторга сообщил о запросах из-за рубежа на поставки Ил-76&quot; -&gt; industrial_production_index, enterprises_count, consumer_price_index.
- &quot;Мировые цены на продовольствие выросли&quot; -&gt; consumer_price_index, industrial_production_index.
- &quot;Более 100 человек обратились в больницу с кишечной инфекцией&quot; -&gt; life_expectancy, hospitals.
- &quot;Введение уголовного наказания за отрицание геноцида&quot; -&gt; crime_count, mortality_rate.
- &quot;КТК сообщил о повреждении причала в результате террористической атаки&quot; -&gt; crime_count, industrial_production_index, enterprises_count; add mortality_rate/life_expectancy only if fatal or health risk is explicit; add air_pollution/wastewater_discharge only if pollution/water channel is explicit.

Relevance:
- 0.90-0.95: main direct/gold-style signal;
- 0.75-0.89: secondary direct signal;
- 0.60-0.74: visible weaker but still direct analytical signal;
- below 0.60: do not return.

Sentiment:
- negative: worsens risk/social condition/pressure;
- positive: improves support, access, production, income, safety, housing, medicine;
- neutral: factual/regulatory without clear improvement/worsening.

Pressure:
- 0.0-0.3: low/no risk pressure;
- 0.4-0.6: moderate;
- 0.7-1.0: high risk/negative pressure.

Return only valid JSON, no markdown, no prose:
{
  &quot;items&quot;: [
    {
      &quot;dataset_row_id&quot;: &quot;{DATASET_ROW_ID}&quot;,
      &quot;exclude&quot;: false,
      &quot;exclude_reason&quot;: &quot;&quot;,
      &quot;no_factor_reason&quot;: &quot;&quot;,
      &quot;annotations&quot;: [
        {
          &quot;factor_id&quot;: &quot;industrial_production_index&quot;,
          &quot;relevance&quot;: 0.9,
          &quot;sentiment&quot;: &quot;neutral&quot;,
          &quot;pressure&quot;: 0.5,
          &quot;confidence&quot;: 0.9,
          &quot;evidence&quot;: &quot;short exact quote from news&quot;,
          &quot;reason&quot;: &quot;visible channel -&gt; factor_id&quot;
        }
      ]
    }
  ]
}
</pre>

</details>


## 3. Запускаем эксперименты и оценку


In [6]:
experiment_runs = pd.DataFrame([
    {
        "step": "v3 social signal",
        "command": ".venv/bin/python scripts/run_v3_social_signal_prompt_experiment.py --variants social_signal_v5 --limit 237 --batch-size 1 --force",
        "output": "analysis_outputs/v3_social_signal_experiment_*.csv",
    },
    {
        "step": "v4 direct / social prompt",
        "command": ".venv/bin/python scripts/run_v4_social_signal_prompt_experiment.py --limit 237 --batch-size 1 --num-ctx 32768 --max-tokens 8192 --force",
        "output": "analysis_outputs/v4_social_signal_experiment_*.csv",
    },
    {
        "step": "old gold evaluation",
        "command": ".venv/bin/python scripts/evaluate_runs_on_old_multilabel_gold.py",
        "output": "analysis_outputs/old_gold_all_runs_metrics.csv",
    },
])
experiment_runs


step,command,output
v3 social signal,.venv/bin/python scripts/run_v3_social_signal_prompt_experiment.py --variants social_signal_v5 --limit 237 --batch-size 1 --force,analysis_outputs/v3_social_signal_experiment_*.csv
v4 direct / social prompt,.venv/bin/python scripts/run_v4_social_signal_prompt_experiment.py --limit 237 --batch-size 1 --num-ctx 32768 --max-tokens 8192 --force,analysis_outputs/v4_social_signal_experiment_*.csv
old gold evaluation,.venv/bin/python scripts/evaluate_runs_on_old_multilabel_gold.py,analysis_outputs/old_gold_all_runs_metrics.csv


## 4. Смотрим журнал запусков и ошибки парсинга


In [7]:
run_status = (
    metrics.groupby(["run_name", "source_file", "run_family", "model_guess"], as_index=False)
    .agg(
        thresholds=("threshold", "nunique"),
        n_news=("n_news", "max"),
        best_micro_f1=("micro_f1", "max"),
        best_recall=("micro_recall", "max"),
        error_rows=("error_rows", "max"),
    )
    .sort_values(["n_news", "best_micro_f1"], ascending=False)
)
run_status.head(20)


run_name,source_file,run_family,model_guess,thresholds,n_news,best_micro_f1,best_recall,error_rows
manual_dataset_prompt_search_best_balanced_recall_v2,manual_dataset_prompt_search_best_balanced_recall_v2.csv,prompt_search,unknown,20,1000,0.4888,0.4536,0
prompt_search_full_balanced_recall_v2_gemma4-e2b_5df03950348d_e98b1a3c5abc,prompt_search_full_balanced_recall_v2_gemma4-e2b_5df03950348d_e98b1a3c5abc.csv,prompt_search,gemma4:e2b,20,1000,0.4888,0.4536,0
manual_dataset_llm_predictions_gemma4-e2b_social-risk-classification-v1_5df03950348d_033f6fd8763b,manual_dataset_llm_predictions_gemma4-e2b_social-risk-classification-v1_5df03950348d_033f6fd8763b.csv,other,gemma4:e2b,20,1000,0.2831,0.223,0
prompt_search_full_production_v1_gemma4-e2b_5df03950348d_b6a2d2f27b85,prompt_search_full_production_v1_gemma4-e2b_5df03950348d_b6a2d2f27b85.csv,prompt_search,gemma4:e2b,20,1000,0.2831,0.223,0
v4_social_signal_experiment_v4_prompt_think_false_v4full965_gemma4_26b_social_signal_v9_f1_balanced_nothink_json_ctx32768_tok16384_b2,v4_social_signal_experiment_v4_prompt_think_false_v4full965_gemma4_26b_social_signal_v9_f1_balanced_nothink_json_ctx32768_tok16384_b2.csv,v4_social_signal,gemma4:26b,20,706,0.6521,0.8321,0
prompt_search_search_balanced_recall_v2_gemma4-e2b_5df03950348d_e98b1a3c5abc,prompt_search_search_balanced_recall_v2_gemma4-e2b_5df03950348d_e98b1a3c5abc.csv,prompt_search,gemma4:e2b,20,300,0.4862,0.43,0
thinking_recall_experiment_latent_candidate_v3_think_true,thinking_recall_experiment_latent_candidate_v3_think_true.csv,thinking_recall,unknown,20,300,0.4802,0.6254,0
thinking_recall_experiment_recall_max_v1_think_true,thinking_recall_experiment_recall_max_v1_think_true.csv,thinking_recall,unknown,20,300,0.4786,0.4365,0
thinking_recall_experiment_recall_max_v1_think_false,thinking_recall_experiment_recall_max_v1_think_false.csv,thinking_recall,unknown,20,300,0.4583,0.4202,0
prompt_search_search_few_shot_major_v3_gemma4-e2b_5df03950348d_0c3d39008fbb,prompt_search_search_few_shot_major_v3_gemma4-e2b_5df03950348d_0c3d39008fbb.csv,prompt_search,gemma4:e2b,20,300,0.4514,0.4235,0


## 5. Пишем код отбора лучших threshold по каждому запуску


In [8]:
def best_threshold_per_run(df, metric="micro_f1", recall="micro_recall", n_col="n_news"):
    work = df[df[metric].notna()].copy()
    work["_n"] = pd.to_numeric(work[n_col], errors="coerce").fillna(0)
    group_col = "run_id" if "run_id" in work.columns else "run_name"
    return (
        work.sort_values([metric, recall, "_n"], ascending=False)
        .groupby(group_col, as_index=False)
        .head(1)
        .drop(columns=["_n"])
    )

best_by_run = best_threshold_per_run(metrics)
pd.DataFrame([
    {"step": "group by run_id", "why": "one experiment has many thresholds"},
    {"step": "sort by micro_f1, recall, n_news", "why": "prefer quality, then coverage"},
    {"step": "keep top row per run", "why": "human-readable comparison table"},
    {"step": "split full vs pilot", "why": "do not compare 30-row pilots directly with 237/706-row runs"},
])


step,why
group by run_id,one experiment has many thresholds
"sort by micro_f1, recall, n_news","prefer quality, then coverage"
keep top row per run,human-readable comparison table
split full vs pilot,do not compare 30-row pilots directly with 237/706-row runs


## 6. Полные прогоны


In [9]:
full_runs = best_by_run[pd.to_numeric(best_by_run["n_news"], errors="coerce").ge(200)].copy()
full_runs = full_runs.sort_values(["micro_f1", "micro_recall", "n_news"], ascending=False)
full_runs[["run_family", "model_guess", "threshold", "n_news", "micro_precision", "micro_recall", "micro_f1", "source_file"]].head(15)


run_family,model_guess,threshold,n_news,micro_precision,micro_recall,micro_f1,source_file
v3_social_signal,qwen3.6-35b:iq3,0.35,237,0.5774,0.7698,0.6599,v3_social_signal_experiment_social_signal_v5_think_false_qwen36_35b_iq3_ctx16k_v5_b1_search237_full.csv
v4_social_signal,gemma4:26b,0.75,706,0.6179,0.6903,0.6521,v4_social_signal_experiment_v4_prompt_think_false_v4full965_gemma4_26b_social_signal_v9_f1_balanced_nothink_json_ctx32768_tok16384_b2.csv
v4_social_signal,gemma4:26b,0.50,237,0.5939,0.6905,0.6385,v4_social_signal_experiment_v4_prompt_think_false_gemma4_26b_ctx32k_tok8k_b1_search237.csv
v4_social_signal,gemma4:26b,0.85,237,0.6192,0.6578,0.6379,v4_social_signal_experiment_v4_prompt_think_false_gemma4_26b_direct_core_targeted_v7_nothink_ctx32768_tok16384_b2_n237.csv
v3_social_signal,gemma4:e4b,0.75,237,0.6125,0.6587,0.6348,v3_social_signal_experiment_social_signal_v5_think_false_gemma4_e4b_v5_b1_search237.csv
v4_social_signal,gemma4:e4b,0.90,237,0.7283,0.56,0.6332,v4_social_signal_experiment_v4_prompt_think_true_gemma4_e4b_direct_core_targeted_v7_think_ctx16384_tok16384_n237.csv
v4_social_signal,gemma4:e4b,0.85,237,0.6311,0.6311,0.6311,v4_social_signal_experiment_v4_prompt_think_true_gemma4_e4b_direct_core_recall_v2_think_ctx16384_tok16384_n237.csv
v4_social_signal,gemma4:e4b,0.90,237,0.6721,0.5467,0.6029,v4_social_signal_experiment_v4_prompt_think_true_gemma4_e4b_direct_core_gold_fewshot_v5_think_ctx16384_tok16384_n237.csv
v4_social_signal,gemma4:e4b,0.85,237,0.6839,0.5289,0.5965,v4_social_signal_experiment_v4_prompt_think_true_gemma4_e4b_direct_core_filtered_v3_think_ctx16384_tok16384_n237.csv
v4_social_signal,gemma4:e4b,0.80,237,0.5794,0.6,0.5895,v4_social_signal_experiment_v4_prompt_think_true_gemma4_e4b_direct_core_gold_calibrated_v4_think_ctx16384_tok16384_n237.csv


## 7. Пилоты отдельно


In [10]:
pilots = best_by_run[pd.to_numeric(best_by_run["n_news"], errors="coerce").between(30, 199, inclusive="both")].copy()
pilots = pilots.sort_values(["micro_f1", "micro_recall", "n_news"], ascending=False)
pilots[["run_family", "model_guess", "threshold", "n_news", "micro_precision", "micro_recall", "micro_f1", "error_rows", "source_file"]].head(12)


run_family,model_guess,threshold,n_news,micro_precision,micro_recall,micro_f1,error_rows,source_file
v3_social_signal,qwen3.6-35b:iq3,0.35,30,0.6364,0.9545,0.7636,0,v3_social_signal_experiment_social_signal_v5_think_false_qwen36_35b_iq3_ctx16k_v5_b1_pilot30.csv
v4_social_signal,qwen35-9b,0.35,41,0.8125,0.7027,0.7536,0,v4_social_signal_experiment_v4_prompt_think_false_v4pilot60_qwen35_9b_high_recall_minimal_v3_nothink_json_ctx32768_tok16384_b2.csv
v3_social_signal,qwen3.6-35b:iq3,0.35,30,0.6,0.9545,0.7368,0,v3_social_signal_experiment_social_signal_v5_think_false_qwen36_35b_iq3_v5_b1_pilot30.csv
v4_social_signal,gemma4:26b,0.35,41,0.7647,0.7027,0.7324,0,v4_social_signal_experiment_v4_prompt_think_false_v4pilot60_gemma4_26b_high_recall_minimal_v3_nothink_json_ctx32768_tok16384_b2.csv
v4_social_signal,gemma4:26b,0.35,41,0.7222,0.7027,0.7123,72,v4_social_signal_experiment_v4_prompt_think_true_v4pilot60_gemma4_26b_high_recall_minimal_v3_think_nojson_guard2_ctx32768_tok8192_b1.csv
v4_social_signal,gemma4:26b,0.85,41,0.8214,0.6216,0.7077,0,v4_social_signal_experiment_v4_prompt_think_false_v4pilot60_gemma4_26b_social_signal_v9_f1_balanced_nothink_json_ctx32768_tok16384_b2.csv
v4_social_signal,gemma4:26b,0.35,78,0.6935,0.7167,0.7049,0,v4_social_signal_experiment_v4_prompt_think_false_v4full965_gemma4_26b_high_recall_minimal_v2_nothink_json_ctx32768_tok16384_b2.partial.csv
v4_social_signal,gemma4:26b,0.35,30,0.6522,0.6818,0.6667,72,v4_social_signal_experiment_v4_prompt_think_true_oldgold_gemma4_26b_high_recall_minimal_v2_think_nojson_guard2_ctx32768_tok8192_b1_pilot30.csv
v4_social_signal,qwen35-9b,0.90,41,0.7037,0.5135,0.5938,0,v4_social_signal_experiment_v4_prompt_think_false_v4pilot60_qwen35_9b_social_signal_v9_f1_balanced_nothink_json_ctx32768_tok16384_b2.csv
v4_social_signal,gemma4:26b,0.70,30,0.4722,0.7727,0.5862,0,v4_social_signal_experiment_v4_prompt_think_true_oldgold_gemma4_26b_direct_core_targeted_v7_think_nojson_guard2_ctx32768_tok8192_b1_pilot30.csv


## 8. Что дали разные семейства экспериментов


In [11]:
by_family = (
    best_by_run.groupby(["run_family", "model_guess"], as_index=False)
    .agg(runs=("run_name", "nunique"), max_n_news=("n_news", "max"), best_micro_f1=("micro_f1", "max"), best_recall=("micro_recall", "max"))
    .sort_values(["best_micro_f1", "best_recall"], ascending=False)
)
by_family.head(25)


run_family,model_guess,runs,max_n_news,best_micro_f1,best_recall
v3_social_signal,qwen3.6-35b:iq3,3,237,0.7636,0.9545
v4_social_signal,qwen35-9b,8,41,0.7536,1.0
v4_social_signal,gemma4:26b,16,706,0.7324,0.8333
v3_social_signal,gemma4:26b,3,18,0.6667,0.75
v3_social_signal,gemma4:e4b,1,237,0.6348,0.6587
v4_social_signal,gemma4:e4b,10,237,0.6332,0.6756
v4_social_signal,unknown,5,237,0.6154,0.6667
v3_social_signal,qwen35-9b,1,237,0.5768,0.5437
v3_social_signal,unknown,10,237,0.4924,0.4484
v4_social_signal,gemma4:e2b,1,237,0.4924,0.4311


## 9. Вывод


In [12]:
summary = pd.DataFrame([
    {"point": "best full old gold micro-F1", "value": round(float(full_runs["micro_f1"].max()), 4)},
    {"point": "best full recall", "value": round(float(full_runs["micro_recall"].max()), 4)},
    {"point": "full runs", "value": len(full_runs)},
    {"point": "practical conclusion", "value": "results are satisfactory for demonstration; keep pilots separate from full runs"},
])
summary


point,value
best full old gold micro-F1,0.6599
best full recall,0.7698
full runs,36
practical conclusion,results are satisfactory for demonstration; keep pilots separate from full runs
